# [1.3.2] Function Vectors & Model Steering (연습 문제)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/12_[1.3.2]_Function_Vectors_&_Model_Steering)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part32_function_vectors_and_model_steering/1.3.2_Function_Vectors_&_Model_Steering_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part32_function_vectors_and_model_steering/1.3.2_Function_Vectors_&_Model_Steering_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널로 보내주시고, 본 장의 내용에 관한 질문은 해당 전용 채널에서 해주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 가는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

> *참고 - 어느 시점에서든 numpy 관련 에러가 발생하면 (예: import 셀을 처음 실행할 때 또는 첫 번째 numpy 함수를 실행할 때), 커널을 재시작하고 설정 코드를 다시 실행해야 합니다. 그러면 에러가 해결될 것입니다.*

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-14-2.png" width="350">

# 소개

이 연습 문제들은 다음과 같은 질문에 대한 탐색 역할을 합니다: ***경사 하강법(gradient descent) 기반이 아닌 방법으로 찾은 벡터를 사용하여 모델의 forward pass에 개입함으로써, 모델이 다른 출력을 생성하거나 다른 동작을 하도록 유도할 수 있을까요?***

대부분의 연습 문제는 [function vectors](https://functions.baulab.info/)에 집중합니다: 이는 in-context learning (ICL) 태스크의 forward pass에서 추출된 벡터들로, zero-shot 프롬프트로부터 해당 태스크의 실행을 유도하기 위해 residual stream에 추가됩니다. 아래 다이어그램이 이를 설명합니다.

<img src="https://functions.baulab.info/images/Paper/fv-demonstrations.png" width="650">

또한 이 연습 문제들을 통해 `nnsight` 라이브러리의 사용법을 익히게 됩니다. 이 라이브러리는 매우 큰 언어 모델, 즉 여러분이 이 과정의 현재 단계에서 익숙해졌을 GPT2-Small보다 더 큰 모델에서 이러한 종류의 작업(및 기타 interpretability 연구)을 지원하도록 설계되었습니다.

마지막 연습 문제 세트에서는 Alex Turner 등의 [steering vectors](https://www.lesswrong.com/posts/5spBue2z2tw4JuDCx/steering-gpt-2-xl-by-adding-an-activation-vector)에 관한 연구를 살펴봅니다. 이는 개념적으로는 관련이 있지만, 목적과 방법론은 다릅니다.

## 콘텐츠 및 학습 목표

### 1️⃣ `nnsight` 소개

이 섹션에서는 `nnsight` 라이브러리를 사용하는 기본 방법인 모델의 forward pass 실행 및 내부 상태 저장 방법을 배웁니다. 또한 `nnsight` 모델로 이어지는 HuggingFace 모델의 기본 사항(예: tokenization 및 모델 출력 처리 방법)에 대해서도 배웁니다.

> ##### 학습 목표
>
> * `nnsight` 라이브러리의 기본 사항과 활용 방안을 배웁니다.
> * HuggingFace 모델의 기본 사항(예: tokenization, 모델 출력)을 배웁니다.
> * 이를 사용하여 GPT-J-6B의 내부 activation을 추출하고 시각화합니다.

### 2️⃣ 태스크를 인코딩하는 hidden states

Function Vectors 논문에서 제기한 다음 질문으로 시작합니다:

> *transformer가 태스크 $T$를 보여주는 예시가 포함된 ICL (in-context-learning) 프롬프트를 처리할 때, 태스크 자체를 인코딩하는 hidden states가 존재하는가?*

우리는 **반의어 태스크(antonym task)**에 대한 ICL 프롬프트 세트로부터 벡터 $h$를 구축하고, 이 벡터로 개입(intervene)하여 모델이 zero-shot 프롬프트에서도 반의어를 생성하게 함으로써 정답이 '예'임을 증명할 것입니다.

이를 위해서는 단순히 activation을 저장하는 것뿐만 아니라, `nnsight`을 사용하여 인과적 개입(causal interventions)을 수행하는 방법을 배워야 합니다.

(참고 - 이 섹션은 구조적으로 function vectors 논문의 2.1절을 따릅니다).

> ##### 학습 목표
>
> * `nnsight`을 사용하여 인과적 개입을 수행하는 방법을 이해하고 직접 수행합니다.
> * function vectors 논문의 "h-vector 결과"를 재현합니다. 즉, residual stream에 태스크를 인코딩하고 zero-shot 프롬프트에서 태스크 동작을 유도할 수 있는 벡터가 포함되어 있음을 확인합니다.

### 3️⃣ Function Vectors

이 섹션에서는 모델의 ICL 성능에 큰 영향을 미치는 attention head 세트를 식별하고, 이 벡터들을 patch 하여 무작위로 섞인 프롬프트에서도 태스크 해결 동작을 유도할 수 있음을 보여줌으로써 논문 결과의 핵심을 재현합니다.

또한 multi-token generation을 위해 `nnsight`을 사용하는 방법과 모델의 동작을 조종(steer)하는 방법을 배웁니다. 국가-수도 태스크와 같은 다양한 태스크에 대해 이를 시도해 볼 수 있는 연습 문제가 있으며, 여기서는 암스테르담에 대해 이야기함으로써 `"When you think of Netherlands, you usually think of"`와 같은 프롬프트를 완성하도록 모델을 조종할 수 있습니다.

(참고 - 이 섹션은 구조적으로 function vectors 논문의 2.2절, 2.3절 및 3절의 일부를 따릅니다).

> ##### 학습 목표
>
> * in-context learning 태스크의 정확한 성능에 각 attention head가 미치는 인과적 영향을 측정하기 위한 지표를 정의합니다.
> * `nnsight` forward pass 중에 모델의 activation을 재배치하여 특정 attention head에 해당하는 activation을 추출하는 방법을 이해합니다.
> * multi-token generation을 위해 `nnsight`을 사용하는 방법을 배웁니다.

### 4️⃣ GPT2-XL의 Steering Vectors

여기서는 이와 다르지만 관련이 있는 연구인 Alex Turner의 steering vectors 연구에 대해 논의합니다. 이 또한 "동작을 변경하기 위해 forward pass(non-SGD) 기반 방법으로 찾은 벡터를 사용하여 residual stream에 개입하는 것"이라는 범주에 속하지만, 설정, 목표 및 접근 방식이 다릅니다.

> ##### 학습 목표
>
> * Alex Turner 등의 steering vectors 연구의 목표와 주요 결과를 이해합니다.
> * 그들의 초기 포스트에 기술된 동작 변화를 재현합니다.

### ☆ 보너스

마지막으로, 현재 매우 활발하게 개발되고 있는 분야인 function vectors 및 steering vectors 연구의 가능한 확장 방향에 대해 논의합니다 (예: 최근 2023년 12월에 발표된 Llama 2-13b steering 관련 논문 등).

## 설정 코드

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import nnsight
except:
    %pip install openai>=1.56.2 nnsight einops jaxtyping plotly transformer_lens==2.17.0 git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python gradio typing-extensions
    %pip install --upgrade pydantic

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import logging
import os
import sys
import time
from collections import defaultdict
from pathlib import Path

import circuitsvis as cv
import einops
import numpy as np
import torch as t
from IPython.display import display
from jaxtyping import Float
from nnsight import CONFIG, LanguageModel
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from torch import Tensor

# Hide some info logging messages from nnsight
logging.disable(sys.maxsize)

t.set_grad_enabled(False)
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part32_function_vectors_and_model_steering"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part32_function_vectors_and_model_steering.solutions as solutions
import part32_function_vectors_and_model_steering.tests as tests
from plotly_utils import imshow

MAIN = __name__ == "__main__"

# 1️⃣ `nnsight` 소개

> ##### 학습 목표
>
> * `nnsight` 라이브러리의 기초와 활용 방안에 대해 배웁니다.
> * HuggingFace 모델의 기초(예: tokenization, 모델 출력)를 배웁니다.
> * 이를 사용하여 GPT-J-6B의 내부 activation을 추출하고 시각화합니다.

## 원격 실행 (Remote execution)

먼저 [remote execution]((https://nnsight.net/notebooks/features/remote_execution/))에 대해 논의하겠습니다. 이는 외부 서버에서 모델을 실행할 수 있는 능력 `nnsight` 이며, 연구 도구로서 이 라이브러리가 가진 주요 장점 중 하나입니다. 이를 통해 사용자의 로컬 머신에서 겪을 수 있는 메모리 및 계산 능력의 한계를 극복할 수 있습니다. 원격 실행이 작동하려면 다음 두 가지가 필요합니다:

1. 커뮤니티 Discord에서 발급받은 API key입니다. [here](https://login.ndif.us/) 에서 요청할 수 있습니다. (다른 적절한 제공자가 없다면 Google을 ID 제공자로 사용하십시오.)
2. 사용하려는 모델이 라이브 상태여야 합니다. 모든 라이브 모델은 상태 페이지 [here](https://nnsight.net/status/) 에서 확인할 수 있습니다.

상태 페이지에서 모든 라이브 모델을 로드하는 데 때때로 약 5분 정도 걸릴 수 있다는 점에 유의하십시오. 아래 드롭다운을 클릭하여 모델이 로드된 후 상태 페이지가 어떻게 보이는지 예시를 확인하시기 바랍니다. 만약 리스트에서 찾는 모델이 보이지 않는다면, 이번 실습을 위해 `REMOTE=False` 를 설정하거나, NDIF Discord에 모델 라이브화를 요청해야 합니다.

<details>
<summary>상태 페이지 예시</summary>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ndif-status.png" width="650">

</details>

## 중요한 구문

여기서는 `nnsight` 모델과 상호작용하기 위한 몇 가지 중요한 구문에 대해 논의하겠습니다. 이 모델들은 HuggingFace 모델의 확장 버전이므로, 일부 정보(예: tokenization)는 일반 HuggingFace 모델과 `nnsight` 모델 모두에 적용되며, 일부 정보(예: forward passes)는 `nnsight`에 특화되어 있습니다. 즉, 표준 HuggingFace 모델만 사용할 경우에는 다르게 작동할 것입니다. 이 차이점을 반드시 염두에 두시기 바랍니다. 그렇지 않으면 구문이 혼란스러울 수 있습니다!

### 모델 설정 (Model config)

각 모델에는 모델에 대한 많은 유용한 정보(예: head 및 layer 수, hidden layer 크기 등)가 포함된 `model.config`이 함께 제공됩니다. `model.config`를 통해 이에 접근할 수 있습니다. 아래 코드를 실행하여 이를 확인하고, 나중에 사용할 유용한 변수들을 정의해 보시기 바랍니다.

In [ ]:
model = LanguageModel("EleutherAI/gpt-j-6b", device_map="auto", dtype=t.bfloat16)
tokenizer = model.tokenizer

N_HEADS = model.config.n_head
N_LAYERS = model.config.n_layer
D_MODEL = model.config.n_embd
D_HEAD = D_MODEL // N_HEADS

print(f"Number of heads: {N_HEADS}")
print(f"Number of layers: {N_LAYERS}")
print(f"Model dimension: {D_MODEL}")
print(f"Head dimension: {D_HEAD}\n")

print("Entire config: ", model.config)

### Tokenizers

모델에는 `model.tokenizer`으로 접근 가능한 tokenizer가 포함되어 있습니다 (TransformerLens와 마찬가지입니다). TransformerLens와 달리, `model.to_str_tokens`과 같은 유틸리티 함수를 사용하지 않고 tokenizer를 직접 사용하겠습니다. 오늘 실습을 위해 중요한 몇 가지 함수는 다음과 같습니다:

* `tokenizer` (즉, 입력값에 대해 함수를 직접 호출하는 것)
    * 문자열(또는 문자열 리스트)을 입력받아 tokenized된 버전을 반환합니다.
    * 딕셔너리를 반환하며, 여기에는 항상 `input_ids` (즉, 실제 token들)이 포함되어 있고, transformer 모델에 특화된 다른 정보들도 포함됩니다 (예: `attention_mask` - 드롭다운 참조).
    * 이 함수의 다른 유용한 인자들:
        * `return_tensors` - 이 값이 `"pt"`이면, 결과가 기본값인 리스트 대신 PyTorch tensor로 반환됩니다.
        * `padding` - True인 경우 (기본값은 False), tokenizer가 가변 길이의 시퀀스를 받아들일 수 있습니다. 짧은 시퀀스는 앞부분에 padding이 추가됩니다 (자세한 내용은 아래 드롭다운 참조).
* `tokenizer.decode`
    * token들을 입력받아 디코딩된 문자열을 반환합니다.
    * 입력이 정수이면 그에 해당하는 문자열을 반환합니다. 입력이 정수 리스트 또는 1D array이면, 해당 문자열들을 모두 연결하여 반환합니다 (이는 때때로 원하는 결과가 아닐 수 있습니다).
* `tokenizer.batch_decode`
    * `tokenizer.decode`과 동일하지만, 문자열을 연결하지 않습니다.
    * 입력이 리스트 또는 1D 정수 array이면 문자열 리스트를 반환합니다. 입력이 2D이면 각 리스트 내부에서 연결을 수행합니다.
* `tokenizer.tokenize`
    * 문자열을 입력받아 문자열 리스트를 반환합니다.

아래 코드를 실행하여 이 함수들이 실제로 어떻게 작동하는지 예시를 확인해 보시기 바랍니다.

In [ ]:
# Calling tokenizer returns a dictionary, containing input ids & other data.
# If returned as a tensor, then by default it will have a batch dimension.
print(tokenizer("This must be Thursday", return_tensors="pt"))

# Decoding a list of integers, into a concatenated string.
print(tokenizer.decode([40, 1239, 714, 651, 262, 8181, 286, 48971, 12545, 13]))

# Using batch decode, on both 1D and 2D input.
print(tokenizer.batch_decode([4711, 2456, 481, 307, 6626, 510]))
print(tokenizer.batch_decode([[1212, 6827, 481, 307, 1978], [2396, 481, 428, 530]]))

# Split sentence into tokens (note we see the special Ġ character in place of prepended spaces).
print(tokenizer.tokenize("This sentence will be tokenized"))

<details>
<summary><code>attention_mask</code>에 관한 참고 사항 (선택 사항)</summary>

`attention_mask`는 1과 0으로 이루어진 시리즈입니다. 우리는 모든 0-position에서 attention을 mask합니다 (즉, 이 token들이 attend되는 것을 허용하지 않습니다). 이는 padding을 해야 할 때 유용합니다. 예를 들어:

```python
model.tokenizer(["Hello world", "Hello"], return_tensors="pt", padding=True)
```

은 다음과 같은 결과를 반환합니다:

```python
{
    'attention_mask': tensor([[1, 1], [0, 1]]),
    'input_ids': tensor([[15496,   995], [50256, 15496]])
}
```

더 짧은 sequence가 앞부분에서 어떻게 padding되었는지 확인할 수 있으며, 이 token에 대한 attention은 mask될 것입니다.

</details>

### Model outputs

높은 수준에서 모델을 실행하는 방법에는 두 가지가 있습니다: `trace` 방법 (단일 forward pass)과 `generate` 방법 (여러 token 생성)입니다. 지금은 `trace`에 집중하고, 나중에 multi-token generation에 대해 다룰 때 `generate`를 논의하겠습니다.

일반적인 HuggingFace 모델에서 forward pass의 기본 동작은 logit(그리고 선택적으로 다른 여러 항목들)을 포함하는 객체를 반환하는 것입니다. `nnsight`의 `trace`의 기본 동작은 아무것도 반환하지 않는 것인데, 이는 우리가 반환하기로 선택한 모든 것이 context manager 내부에서 명시적으로 반환되기 때문입니다.

아래는 모델을 실행(하고 모델의 내부 상태에 접근)하는 가장 간단한 코드 예시입니다. 코드를 실행하여 출력을 확인한 다음, 아래의 설명을 읽어보시기 바랍니다. remote execution을 사용하는 경우, 먼저 API key를 획득하고 설정하는 것을 잊지 마십시오!

In [ ]:
# If you have an API key & want to work remotely, then set REMOTE = True and replace "YOUR-API-KEY"
# with your actual key. If not, then leave REMOTE = False.
REMOTE = False
if REMOTE:
    CONFIG.set_default_api_key("YOUR-API-KEY")

prompt = "The Eiffel Tower is in the city of"

with model.trace(prompt, remote=REMOTE):
    # Save the model's hidden states
    hidden_states = model.transformer.h[-1].output[0].save()

    # Save the model's logit output
    logits = model.lm_head.output[0, -1].save()

# Get the model's logit output, and it's next token prediction
print(f"logits.shape = {logits.shape} = (vocab_size,)")
print("Predicted token ID =", predicted_token_id := logits.argmax().item())
print(f"Predicted token = {tokenizer.decode(predicted_token_id)!r}")

# Print the shape of the model's residual stream
print(f"\nresid.shape = {hidden_states.shape} = (batch_size, seq_len, d_model)")

하나씩 자세히 살펴보겠습니다.

**먼저, 모델 객체에서 `.trace(...)`를 호출하여 context block을 생성합니다.** 이는 특정 prompt가 주어졌을 때 token을 생성하겠다는 것을 의미합니다.

```python
with model.trace(prompt, remote=REMOTE):
```

기본적으로 이를 실행하면 모델이 로컬에서 로드 및 실행되지만, `remote=REMOTE`를 전달하면 모델이 서버에서 실행됩니다. 이는 머신에 담기에 너무 큰 모델을 사용할 때 매우 유용합니다 (또는 머신에 들어갈 수는 있지만 크기 때문에 느리게 실행되는 모델의 경우에도 유용합니다. 하지만 충분히 큰 GPU에서 이 자료를 실행하고 있다면 `REMOTE=False`로 설정하는 것이 좋을 수 있습니다). 입력 인자는 문자열, token 리스트, token 텐서 등 다양한 형식을 가질 수 있습니다. 여기서는 단순히 문자열 `prompt`를 사용했습니다.

`nnsight`에서 가장 흥미로운 부분은 모델의 내부 상태에 접근할 수 있는 능력입니다 (TransformerLens에서 이미 해보셨을 수도 있습니다). 이제 이것이 어떻게 작동하는지 살펴보겠습니다!

```python
hidden_states = model.transformer.h[-1].output[0].save()
```

이 라인에서 우리는 다음과 같이 말하고 있습니다: forward pass 내에서, transformer의 마지막 레이어 `model.transformer.h[-1]`에 접근하고, 이 레이어의 출력 `.output` (텐서들의 튜플입니다)에 접근하여, 이 튜플의 첫 번째 텐서를 인덱싱 `.output[0]`하고, 이를 저장 `.save()`합니다.

이 라인을 조금 더 자세히 분석해 보겠습니다:

* `model.transformer.h[-1]`은 우리 transformer의 모듈입니다.
    * `print(model)`를 하면, 이것이 `transformer`과 `lm_head` ("language modelling head"의 약자)로 구성되어 있음을 알 수 있습니다. `transformer` 모듈은 embedding과 dropout, 일련의 레이어들 (`.h`, "hidden states"의 약자), 그리고 최종 layernorm으로 이루어져 있습니다. 따라서 `.h[-1]`를 인덱싱하면 최종 레이어를 얻게 됩니다.
    * 참고 - 작업 중인 모델의 문서 페이지를 방문하는 것이 유용할 때가 많습니다. 예를 들어, GPT-J [here](https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html)를 찾을 수 있습니다. 모든 모델이 TransformerLens에서 익숙했던 것처럼 깔끔하고 균일한 표준 아키텍처를 가지고 있지는 않습니다!
* `.output[0]`는 이 모듈의 출력을 **proxy** 형태로 제공합니다.
    * 모듈의 출력은 종종 튜플입니다 (마찬가지로 [documentation page](https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html)에서 각 모듈의 출력이 무엇인지 확인할 수 있습니다). 이 경우, 2개의 텐서로 이루어진 튜플이며, 그 중 첫 번째가 실제 레이어 출력 (우리가 원하는 것)입니다.
    * proxy에 대해 연산을 수행해도 여전히 proxy가 반환됩니다. 이것이 우리가 `output` proxy 튜플을 인덱싱하여 proxy 텐서를 얻을 수 있는 이유입니다!
* `.save()`는 이 proxy 출력을 받아 실제 객체를 반환합니다 (이제 context manager 외부에서 접근할 수 있습니다).

<details>
<summary><code>save</code> (선택 사항)</summary>에 대한 조금 더 자세한 설명입니다.

더 구체적으로, `.save()`는 **intervention computational graph**에 proxy의 값을 복제하도록 알려주어, forward pass가 끝난 후에도 proxy의 값에 접근할 수 있게 합니다.

우리가 구축하고 있는 intervention computational graph를 처리하는 동안, proxy의 값이 더 이상 필요하지 않게 되면 그 값은 참조 해제되어 파괴됩니다. 만약 이를 저장했다면, 이 일이 일어난 후에도 (즉, context manager 외부에서) proxy의 값에 접근할 수 있습니다.

</details>

<details>
<summary>선택 과제 - <code>.output</code>가 2개의 텐서 튜플을 반환한다고 언급했습니다. <a href="https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html">문서 페이지</a>를 사용하여 이 튜플의 두 번째 텐서가 무엇인지 알 수 있을까요?</summary>

두 번째 출력 또한 길이가 2인 텐서 튜플입니다. GPT-J 소스 코드에서 이것들은 `present`라고 불립니다. 이것들은 이번 forward pass에서 계산된 key와 value를 나타냅니다 (이전 forward pass에서 계산되어 모델에 캐시된 것들과 대조적입니다). 우리는 단 하나의 새로운 token만 생성하고 있으므로, 이것들은 단순히 전체 key와 value입니다.

</details>

<br>

다음 명령어:

```python
logits = model.lm_head.output[0, -1].save()
```

는 매우 유사한 방식으로 이해할 수 있습니다. 유일한 차이점은 language modelling head (즉, 맨 끝의 unembedding)인 `lm_head`의 출력에 접근한다는 것이며, 출력은 텐서 튜플이 아니라 `(batch, seq, d_vocab)` 모양의 단일 텐서라는 점입니다. 이 부분에 대해서도 다시 [documentation page](https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html)를 확인하십시오.

Hugging Face 모델을 사용해 보셨다면 모델 출력에서 logit을 직접 얻는 것에 익숙하시겠지만, 여기서는 일반적으로 다른 activation과 마찬가지로 모델 내부에서 logit을 추출합니다. 왜냐하면 이를 통해 **우리가 무엇을 반환할지 정확하게 제어**할 수 있기 때문입니다. 매우 큰 텐서를 많이 반환하면 서버에서 다운로드하는 데 시간이 꽤 걸릴 수 있습니다 (transformer의 경우 `d_vocab`가 보통 50k 정도로 매우 크다는 점을 기억하십시오). 이에 대한 더 자세한 논의는 아래의 "어떤 객체를 저장할 것인가" 섹션을 참조하십시오.

### Output vs input

`.input` 또는 `.inputs`를 사용하여 모듈의 입력 또한 추출할 수 있습니다. 모듈의 forward 메서드가 `module.forward(*args, **kwargs)`로 호출된다면, `.inputs`는 `(tuple_of_args, dict_of_kwargs)`의 튜플을 반환합니다. 또는, `.input`은 `.inputs[0][0]`의 별칭입니다. 즉, 모듈의 forward 메서드에서 첫 번째 인자 (보통 우리가 원하는 텐서)를 반환합니다.

확신이 서지 않을 때는 `print(module.input.shape)`로 디버깅할 수 있다는 점을 기억하십시오. `.inputs`이 입력 튜플이라 하더라도, 에러를 일으키는 대신 튜플 내 모든 텐서의 모양을 재귀적으로 출력해 줄 것입니다.

### 어떤 객체를 저장할 것인가

위에서 우리는 길이가 50k인 벡터인 `logits`를 저장했습니다. 일반적으로는 가능한 한 작은 객체를 저장하는 것이 가장 좋습니다. 그래야 서버에서 다운로드해야 할 객체의 크기가 줄어들기 때문입니다. 예를 들어, 다음 token completion만 원한다면 logit에 argmax를 취한 후 그 결과만 저장하십시오! 모든 기본적인 텐서 연산은 context manager 내에서 수행할 수 있습니다.

## 이를 실제로 적용하기

### 연습 문제 - attention head 시각화하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

방금 많은 내용을 다루었으므로, 이를 실제로 적용해 보겠습니다. 첫 번째 과제는 transformer의 0번째 layer에서 attention pattern을 추출하고, circuitsvis를 사용하여 이를 시각화하는 것입니다. 참고로, circuitsvis의 문법은 다음과 같습니다:

```python
cv.attention.attention_patterns(
    tokens=tokens,
    attention=attention,
)
```

여기서 `tokens`는 문자열 리스트이며, `attention`은 `(num_heads, num_tokens, num_tokens)` 형태의 tensor입니다.

진행하다가 막힌다면, GPT-J의 소스 코드 [here's a link](https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html)를 확인하십시오. `GPTJAttention` 블록 내에서 attention pattern이 어떻게 계산되는지 찾아보시기 바랍니다.

*참고 - 위 링크의 소스 코드를 보시면 알 수 있듯이, 이 모델은 attention probability에 dropout을 사용합니다. inference mode에서는 dropout이 비활성화되므로(그리고 `generate` 메서드를 사용하면 모델이 항상 inference mode가 됩니다), 모델의 동작에는 영향을 주지 않습니다. 하지만 이는 여전히 모델에 존재하는 layer이므로, 다른 모듈과 마찬가지로 입력이나 출력에 접근할 수 있습니다.*

<details>
<summary>여담 - inference mode</summary>

Dropout은 inference mode에서 동작이 변하는 두 가지 주요 layer 중 하나입니다 (다른 하나는 BatchNorm입니다).

inference mode 없이 모델을 실행하고 싶다면, 코드를 `with model.trace(inference=False):`로 감싸면 됩니다. 하지만 이 연습 문제들의 목적을 위해서는 이에 대해 걱정하실 필요 없습니다.

</details>

적절한 모듈을 참조하는 방법에 대해 어려움이 있다면, 다음 힌트를 참고하십시오:

<details>
<summary>힌트 - 어떤 모듈에서 attention을 가져와야 하는가</summary>

`model.transformer.h[0].attn.attn_dropout.input`에서 attention을 추출해야 합니다. `.output`을 사용해도 동일한 값을 얻을 수 있습니다 (다만 dummy batch dimension으로 인해 차이가 있을 수 있습니다). dropout layer는 하나의 입력만 받고 하나의 출력만 반환하므로, 두 방법 모두 단일 tensor를 반환합니다.

</details>

<details>
<summary>여담 - GPT2 tokenizer는 공백을 나타내기 위해 특수 문자를 사용합니다 </summary>

GPT2 tokenizer는 앞에 붙는 공백을 나타내기 위해 "Ġ"를 사용합니다. 따라서 ["My", " name", " is", " James"]는 ["My", "Ġname", "Ġis", "ĠJames"]로 tokenized 됩니다. "Ġ"를 실제 공백으로 교체했는지 확인하십시오.

</details>

In [ ]:
# YOUR CODE HERE - extract and visualize attention

<details>
<summary>솔루션 (및 설명)</summary>

```python
with model.trace(prompt, remote=REMOTE):
    attn_patterns = model.transformer.h[0].attn.attn_dropout.input.save()

# Get string tokens (replacing special character for spaces)
str_tokens = model.tokenizer.tokenize(prompt)
str_tokens = [s.replace('Ġ', ' ') for s in str_tokens]

# Attention patterns (squeeze out the batch dimension)
attn_patterns_value = attn_patterns.squeeze(0)

print("Layer 0 Head Attention Patterns:")
display(cv.attention.attention_patterns(
    tokens=str_tokens,
    attention=attn_patterns_value,
))
```

설명:

* context manager 내부에서:
    * `attn_dropout`의 입력을 가져와 attention pattern에 접근합니다.
        * GPT-J 소스 코드에서, attention weight는 key와 query 벡터로부터 표준 torch 함수(및 이름 없는 `nn.Softmax` 모듈)를 통해 계산되며, attention layer 출력을 계산하기 전에 dropout layer를 통과하는 것을 볼 수 있습니다. 따라서 dropout layer의 입력에 접근함으로써, dropout이 적용되기 전의 attention weight를 얻을 수 있습니다.
        * 추론 모드에서는 dropout이 작동하지 않는다는 앞서 논의한 점 때문에, `attn_dropout`의 출력을 사용해도 동일한 값을 얻을 수 있습니다.
    * `.save()` 메서드를 사용하여 attention pattern을 (객체 형태로) 저장합니다.
* context manager 외부에서:
    * `tokenize` 메서드를 사용하여 prompt를 tokenize합니다.
        
</details>

선택 사항인 보너스 연습 문제로, key와 query 벡터를 사용하여 처음부터 직접 계산함으로써 이것들이 올바른 attention pattern인지 직접 확인할 수 있습니다. `model.transformer.h[0].attn.q_proj.output`을 사용하면 query 벡터를, `k_proj`을 사용하면 key 벡터를 얻을 수 있습니다. 하지만 주의해야 할 점은 GPT-J가 **rotary embeddings**를 사용한다는 것이며, 이로 인해 key와 query로부터 attention pattern을 계산하는 것이 평소보다 조금 더 어렵습니다. rotary embeddings에 대한 심층적인 논의는 [here](https://blog.eleuther.ai/rotary-embeddings/)를, 대략적인 직관은 [here](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=bef36Bf9k7FYsCt1DpzCw6eV)을 참고하시기 바랍니다.

# 2️⃣ Task-encoding hidden states

> ##### 학습 목표
>
> * `nnsight`을 사용하여 causal intervention을 수행하는 방법을 이해하고, 직접 수행해 봅니다.
> * function vectors 논문의 "h-vector 결과"를 재현합니다. 즉, residual stream에 task를 인코딩하는 벡터가 포함되어 있으며, 이것이 zero-shot prompt에서 task 동작을 유도할 수 있음을 확인합니다.

우리는 Function Vectors 논문에서 제기된 다음 질문으로 시작합니다:

> *transformer가 태스크 $T$를 보여주는 예시들이 포함된 ICL (in-context-learning) 프롬프트를 처리할 때, hidden state 중 태스크 자체를 인코딩하는 것이 있을까요?*

우리는 **antonym task**를 위한 ICL 프롬프트 세트로부터 벡터 $h$를 구축하고, 이 벡터로 intervention을 수행하여 모델이 zero-shot 프롬프트에서도 반의어를 생성하게 함으로써 그 답이 '예'임을 증명할 것입니다.

이를 위해서는 단순히 activation을 저장하는 것뿐만 아니라, `nnsight`를 사용하여 causal intervention을 수행하는 방법을 배워야 합니다.

참고 - 이 섹션은 구조적으로 function vectors 논문의 2.1절을 따릅니다.

## ICL Task

### 연습 문제 (선택 사항) - 자신만의 반의어 쌍 생성하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> If you choose to do this exercise, you should spend up to 10-30 minutes on it - depending on your familiarity with the OpenAI Python API.
> ```

이 연습 문제에서 사용할 반의어 데이터셋을 위해 두 가지 옵션을 제공합니다.

1. 첫째, `data/antonym_pairs.txt` 파일에 단어 쌍 목록을 제공했습니다.
2. 둘째, 이 논문과 같은 실험을 수행하고 싶다면, GPT-4나 다른 모델로부터 프롬프트를 생성하는 방법을 배우는 것이 좋은 연습이 될 수 있습니다 (이 방식이 우리가 이 연습 문제를 위한 데이터를 생성한 방법입니다).

제공된 단어 목록을 그대로 사용하고 싶다면, 이 연습 문제를 건너뛰고 아래 코드를 실행하여 텍스트 파일에서 데이터셋을 로드하십시오. 반대로, 자신만의 데이터셋을 생성하고 싶다면 아래의 `generate_dataset` 함수를 작성하여 GPT-4에 쿼리를 보내 반의어 쌍 목록을 가져오면 됩니다.

chat completions API 사용법을 아직 모르신다면 [here](https://platform.openai.com/docs/guides/gpt/chat-completions-api) 가이드를 참조하십시오. 안내가 필요하다면 아래의 두 드롭다운을 (순서대로) 사용하십시오.

<details>
<summary>시작하기 #1</summary>

권장하는 템플릿은 다음과 같습니다:

```python
client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": antonym_task},
        {"role": "assistant", "content": start_of_response},
    ]
)
```

여기서 `antonym_task`은 반의어 과업을 설명하며, `start_of_respose`는 모델의 이후 동작을 유도하기 위해 시작 프롬프트(예: "Sure, here are some antonyms: ...")를 제공합니다.

</details>

<details>
<summary>시작하기 #2</summary>

실제 요청에 사용할 수 있는 템플릿은 다음과 같습니다:

```python
example_antonyms = "old: young, top: bottom, awake: asleep, future: past, "

response = openai.ChatCompletion.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"Give me {N} examples of antonym pairs. They should be obvious, i.e. each word should be associated with a single correct antonym."},
        {"role": "assistant", "content": f"Sure! Here are {N} pairs of antonyms satisfying this specification: {example_antonyms}"},
    ]
)
```

여기서 `N`는 함수 인자입니다. 몇 가지 반의어 예시를 제공하고 이를 GPT-4의 completion 시작 부분에 추가했다는 점에 유의하십시오. 이는 나머지 출력 내용을 유도하기 위한 전형적인 기법입니다 (실제로 adversarial attacks에서 흔히 사용됩니다).

</details>

참고 - 반의어로 반환된 모든 단어를 GPT-J가 해결할 수 있는 것은 아닐 수 있습니다. 이 섹션에서는 이 부분에 대해 너무 걱정하지 않겠습니다. zero-shot intervention을 테스트할 때는 GPT-J가 실제로 해결할 수 있는 사례들만 사용하도록 하겠습니다.

In [ ]:
def generate_antonym_dataset(N: int):
    """
    Generates 100 pairs of antonyms, in the form of a list of 2-tuples.
    """
    assert os.environ.get("OPENAI_API_KEY", None) is not None, "Please set your API key before running this function!"

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": f"Generate {N} pairs of antonyms in the form of a list of 2-tuples. For example, [['old', 'young'], ['top', bottom'], ['awake', 'asleep']...].",
            },
            {"role": "assistant", "content": "Sure, here is a list of 100 antonyms: "},
        ],
    )
    return response


if os.environ.get("OPENAI_API_KEY", None) is not None:
    ANTONYM_PAIRS = generate_antonym_dataset(100)
    # Save the word pairs in a text file
    with open(section_dir / "data" / "my_antonym_pairs.txt", "w") as f:
        for word_pair in ANTONYM_PAIRS:
            f.write(f"{word_pair[0]} {word_pair[1]}\n")

# Load the word pairs from the text file
with open(section_dir / "data" / "antonym_pairs.txt", "r") as f:
    ANTONYM_PAIRS = [line.split() for line in f.readlines()]

print(ANTONYM_PAIRS[:10])

## ICL 데이터셋

이 단어 쌍 리스트를 처리하기 위해, 몇 가지 유용한 클래스를 제공해 드립니다.

먼저, `ICLSequence` 클래스가 있으며, 이 클래스는 단어 쌍 리스트를 입력으로 받아 이 단어들로부터 prompt(및 completion)를 구성하는 메서드들을 포함하고 있습니다. 아래 코드를 실행하여 어떻게 작동하는지 확인해 보시기 바랍니다.

In [ ]:
class ICLSequence:
    """
    Class to store a single antonym sequence.

    Uses the default template "Q: {x}\nA: {y}" (with separate pairs split by "\n\n").
    """

    def __init__(self, word_pairs: list[list[str]]):
        self.word_pairs = word_pairs
        self.x, self.y = zip(*word_pairs)

    def __len__(self):
        return len(self.word_pairs)

    def __getitem__(self, idx: int):
        return self.word_pairs[idx]

    def prompt(self):
        """Returns the prompt, which contains all but the second element in the last word pair."""
        p = "\n\n".join([f"Q: {x}\nA: {y}" for x, y in self.word_pairs])
        return p[: -len(self.completion())]

    def completion(self):
        """Returns the second element in the last word pair (with padded space)."""
        return " " + self.y[-1]

    def __str__(self):
        """Prints a readable string representation of the prompt & completion (indep of template)."""
        return f"{', '.join([f'({x}, {y})' for x, y in self[:-1]])}, {self.x[-1]} ->".strip(", ")


word_list = [["hot", "cold"], ["yes", "no"], ["in", "out"], ["up", "down"]]
seq = ICLSequence(word_list)

print("Tuple-representation of the sequence:")
print(seq)
print("\nActual prompt, which will be fed into the model:")
print(seq.prompt())

둘째로, `ICLDataset` 클래스가 있습니다. 이 클래스 역시 단어 쌍 리스트를 입력으로 받으며, prompt와 completion의 batch를 생성하는 메서드들을 가지고 있습니다. 이 클래스는 clean prompt(각 쌍이 실제로 반의어 쌍인 경우)와 corrupted prompt(각 쌍의 정답이 데이터셋에서 무작위로 선택된 경우)를 모두 생성할 수 있습니다.

In [ ]:
class ICLDataset:
    """
    Dataset to create antonym pair prompts, in ICL task format. We use random seeds for consistency
    between the corrupted and clean datasets.

    Inputs:
        word_pairs:
            list of ICL task, e.g. [["old", "young"], ["top", "bottom"], ...] for the antonym task
        size:
            number of prompts to generate
        n_prepended:
            number of antonym pairs before the single-word ICL task
        bidirectional:
            if True, then we also consider the reversed antonym pairs
        corrupted:
            if True, then the second word in each pair is replaced with a random word
        seed:
            random seed, for consistency & reproducibility
    """

    def __init__(
        self,
        word_pairs: list[list[str]],
        size: int,
        n_prepended: int,
        bidirectional: bool = True,
        seed: int = 0,
        corrupted: bool = False,
    ):
        assert n_prepended + 1 <= len(word_pairs), "Not enough antonym pairs in dataset to create prompt."

        self.word_pairs = word_pairs
        self.word_list = [word for word_pair in word_pairs for word in word_pair]
        self.size = size
        self.n_prepended = n_prepended
        self.bidirectional = bidirectional
        self.corrupted = corrupted
        self.seed = seed

        self.seqs = []
        self.prompts = []
        self.completions = []

        # Generate the dataset (by choosing random word pairs, and constructing ICLSequence objects)
        for n in range(size):
            np.random.seed(seed + n)
            random_pairs = np.random.choice(len(self.word_pairs), n_prepended + 1, replace=False)
            # Randomize the order of each word pair (x, y).
            # If not bidirectional, we always have x -> y not y -> x
            random_orders = np.random.choice([1, -1], n_prepended + 1)
            if not (bidirectional):
                random_orders[:] = 1
            word_pairs = [self.word_pairs[pair][::order] for pair, order in zip(random_pairs, random_orders)]
            # If corrupted, then replace y with a random word in all (x, y) pairs except the last one
            if corrupted:
                for i in range(len(word_pairs) - 1):
                    word_pairs[i][1] = np.random.choice(self.word_list)
            seq = ICLSequence(word_pairs)

            self.seqs.append(seq)
            self.prompts.append(seq.prompt())
            self.completions.append(seq.completion())

    def create_corrupted_dataset(self):
        """Creates a corrupted version of the dataset (with same random seed)."""
        return ICLDataset(
            self.word_pairs,
            self.size,
            self.n_prepended,
            self.bidirectional,
            corrupted=True,
            seed=self.seed,
        )

    def __len__(self):
        return self.size

    def __getitem__(self, idx: int):
        return self.seqs[idx]

이 데이터셋이 어떻게 작동하는지 아래에서 확인하실 수 있습니다. **정답 completion 앞에는 공백이 추가되어 있다는 점에 유의하십시오**. 이는 antonym prompt가 구성되는 방식이며, 정답은 `"A: answer" -> ["A", ":", " answer"]` 로 tokenized 됩니다. 앞의 공백을 잊어버리는 것은 transformer를 다룰 때 흔히 발생하는 실수입니다!

In [ ]:
dataset = ICLDataset(ANTONYM_PAIRS, size=10, n_prepended=2, corrupted=False)

table = Table("Prompt", "Correct completion")
for seq, completion in zip(dataset.seqs, dataset.completions):
    table.add_row(str(seq), repr(completion))

rprint(table)

이 출력을 `corrupted=True` 일 때의 결과와 비교해 보십시오. 프롬프트에서 마지막 쌍을 *제외한* 각 쌍의 두 번째 요소는 무작위 요소로 대체되었지만(예: `(right, left)` 이 `(right, pivate)` 로 변경), 마지막 쌍은 변경되지 않은 상태로 유지됩니다. 이는 모델이 쌍들이 어떤 패턴을 따르고 있는지 추론하는 능력을 효과적으로 파괴할 것입니다.

In [ ]:
dataset = ICLDataset(ANTONYM_PAIRS, size=10, n_prepended=2, corrupted=True)

table = Table("Prompt", "Correct completion")
for seq, completions in zip(dataset.seqs, dataset.completions):
    table.add_row(str(seq), repr(completions))

rprint(table)

<details>
<summary>참고 - <code>rich</code> 라이브러리</summary>

`rich` 라이브러리는 Python notebook이나 터미널에서 출력을 더 명확하게 표시해 주는 유용한 작은 라이브러리입니다. 이 워크숍에 필수적인 것은 아니지만, 도구 상자에 갖춰두면 좋은 유용한 도구입니다.

가장 중요한 함수는 `rich.print` (보통 `rprint`로 임포트합니다) 입니다. 이 함수는 기본적인 문자열을 출력할 수 있을 뿐만 아니라, 색상 출력을 위해 다음과 같은 구문을 지원합니다:

```python
rprint("[green]This is green text[/], this is default color")
```

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/rprint-1.png" width="350">

또한 텍스트를 굵게 또는 밑줄 쳐서 표시하기 위해 다음을 사용할 수 있습니다:

```python
rprint("[u dark_orange]This is underlined[/], and [b cyan]this is bold[/].")
```

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/rprint-2.png" width="350">

표(table)를 출력할 수도 있습니다:

```python
from rich.table import Table

table = Table("Col1", "Col2", title="Title") # title is optional
table.add_row("A", "a")
table.add_row("B", "b")

rprint(table)
```

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/rprint-3.png" width="150">

텍스트 서식(굵게, 밑줄, 색상 등)은 표의 셀 내부에서도 지원됩니다.

</details>

## Task-encoding vector

### 연습 문제 - antonym 데이터셋에 대한 forward pass

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

아래의 `calculate_h` 함수를 완성해야 합니다. 이 함수는 다음과 같은 동작을 수행해야 합니다:

* 이전에 설명드린 `nnsight` 문법을 사용하여, 데이터셋 프롬프트(즉, `.prompts` 속성)로 모델의 forward pass를 실행합니다.
* 모델의 출력(즉, 배치의 각 프롬프트에 대한 string-token completion 리스트)과 layer `layer` 끝에서의 residual stream 값(예를 들어 `layer = -1`인 경우, logit으로 변환하기 전 residual stream의 최종 값)을 튜플로 반환합니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/h-intervention-1.png" width="900">

각 프롬프트의 가장 마지막 시퀀스 위치, 즉 모델이 antonym 예측을 수행하는 마지막 `-1` token에 대한 residual stream 값과 completion 값만 반환해야 합니다.

<details>
<summary> 도움말 - 입력 배치를 실행하고 인덱싱하는 방법을 잘 모르겠습니다.</summary>

`generator.invoke` 함수에 문자열 리스트를 전달하면, 자동으로 padding이 적용되어 tokenization됩니다.

적용되는 padding 방식은 **left padding**입니다. 즉, 시퀀스 위치 `-1`에서 인덱싱하면, 프롬프트들의 길이가 서로 다르더라도 리스트 내 모든 프롬프트의 마지막 token을 가져오게 됩니다.

</details>

In [ ]:
def calculate_h(model: LanguageModel, dataset: ICLDataset, layer: int = -1) -> tuple[list[str], Tensor]:
    """
    Averages over the model's hidden representations on each of the prompts in `dataset` at layer
    `layer`, to produce a single vector `h`.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        dataset: ICLDataset
            the dataset whose prompts `dataset.prompts` you're extracting the activations from (at
            the last seq pos)
        layer: int
            the layer you're extracting activations from

    Returns:
        completions: list[str]
            list of the model's next-token predictions (i.e. the strings the model predicts to
            follow the last token)
        h: Tensor
            average hidden state tensor at final sequence position, of shape (d_model,)
    """
    raise NotImplementedError()


tests.test_calculate_h(calculate_h, model)

<details><summary>솔루션</summary>

```python
def calculate_h(model: LanguageModel, dataset: ICLDataset, layer: int = -1) -> tuple[list[str], Tensor]:
    """
    Averages over the model's hidden representations on each of the prompts in `dataset` at layer
    `layer`, to produce a single vector `h`.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        dataset: ICLDataset
            the dataset whose prompts `dataset.prompts` you're extracting the activations from (at
            the last seq pos)
        layer: int
            the layer you're extracting activations from

    Returns:
        completions: list[str]
            list of the model's next-token predictions (i.e. the strings the model predicts to
            follow the last token)
        h: Tensor
            average hidden state tensor at final sequence position, of shape (d_model,)
    """
    with model.trace(dataset.prompts, remote=REMOTE):
        h = model.transformer.h[layer].output[0][:, -1].mean(dim=0).save()
        logits = model.lm_head.output[:, -1]
        next_tok_id = logits.argmax(dim=-1).save()

    completions = model.tokenizer.batch_decode(next_tok_id)
    return completions, h
```
</details>

모델의 antonym 데이터셋 출력 결과를 표시하고, 모델의 예측이 정답인 예시를 강조해 주는 helper 함수를 제공해 드렸습니다. 많은 completion이 줄바꿈으로 이루어져 있기 때문에, 이를 더 명확하게 확인하기 위해 `repr` 함수를 사용하고 있다는 점에 유의하시기 바랍니다!

antonym 데이터셋이 잘 구축되었다면, 대부분의 경우 모델의 completion이 정답이며, 대부분의 실수는 단순 복사(예: `wet -> dry` 대신 `wet -> wet`을 예측)이거나 실제로는 실수로 간주되지 않아야 할 이해 가능한 completion(예: `right -> wrong` 대신 `right -> left`을 예측)일 것입니다. 엄격하게 접근한다면, 모델이 작업을 정확하게 수행할 수 있는 예시만 포함되도록 이 데이터셋을 필터링해야 하겠지만, 이번 실습에서는 이 부분은 고려하지 않겠습니다.

In [ ]:
def display_model_completions_on_antonyms(
    model: LanguageModel,
    dataset: ICLDataset,
    completions: list[str],
    num_to_display: int = 20,
) -> None:
    table = Table(
        "Prompt (tuple representation)",
        "Model's completion\n(green=correct)",
        "Correct completion",
        title="Model's antonym completions",
    )

    for i in range(min(len(completions), num_to_display)):
        # Get model's completion, and correct completion
        completion = completions[i]
        correct_completion = dataset.completions[i]
        correct_completion_first_token = model.tokenizer.tokenize(correct_completion)[0].replace("Ġ", " ")
        seq = dataset.seqs[i]

        # Color code the completion based on whether it's correct
        is_correct = completion == correct_completion_first_token
        completion = f"[b green]{repr(completion)}[/]" if is_correct else repr(completion)

        table.add_row(str(seq), completion, repr(correct_completion))

    rprint(table)


# Get uncorrupted dataset
dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=2)

# Getting it from layer 12, as in the description in section 2.1 of paper
model_completions, h = calculate_h(model, dataset, layer=12)

# Displaying the output
display_model_completions_on_antonyms(model, dataset, model_completions)

### 여러 번의 invoke 사용하기

`nnsight`의 또 다른 멋진 기능은 여러 개의 서로 다른 batch를 모델에 한 번에 실행하거나(또는 동일한 batch를 여러 번 실행), 이를 통해 causal intervention을 수행하기 위한 매우 깔끔한 구문을 사용할 수 있다는 점입니다. 다음과 같이 작성하는 대신:

```python
with model.trace(inputs, remote=REMOTE):
    # some causal interventions
```

이중 중첩 context manager를 작성할 수 있습니다:

```python
with model.trace(remote=REMOTE) as tracer:
    with tracer.invoke(inputs):
        # some causal interventions
    
    with tracer.invoke(other_inputs):
        # some other causal interventions
```

두 입력은 병렬로 함께 실행되며, 하나의 `tracer.invoke` 블록 내에서 정의된 proxy는 다른 블록에서도 사용할 수 있습니다. 일반적인 사용 사례는 clean 입력과 corrupted 입력을 준비하여, 한쪽에서 다른 쪽으로 patch를 수행하고 단 한 번의 forward pass로 두 출력을 모두 얻는 것입니다:

```python
with model.trace(remote=REMOTE) as tracer:
    with tracer.invoke(clean_inputs):
        # extract clean activations
        clean_activations = model.transformer.h[10].output[0]
    
    with tracer.invoke(corrupted_inputs):
        # patch clean into corrupted
        model.transformer.h[10].output[0][:] = clean_activations
```

나중 연습 문제에서 이와 유사한 작업을 수행하게 됩니다. 하지만 첫 번째 연습 문제(바로 아래)에서는 context manager 외부에서 정의된 벡터들로만 intervention을 수행하게 됩니다.

**주의해야 할 중요한 점** - proxy가 정의되기 전에 사용하지 않도록 주의하십시오! 예를 들어, `model.transformer.h[10]`에서 `clean_activations`를 추출한 다음 이를 `model.transformer.h[9]`에 intervention 하려는 경우, 이는 병렬로 처리될 수 없습니다 (먼저 clean activation을 추출한 *후에* patched forward pass를 실행해야 합니다). 이렇게 하면 경고 메시지가 표시되어야 하지만, 일부 경우에는 아무런 표시 없이 통과될 수 있으므로 각별히 주의해야 합니다!

### 연습 문제 - $h$로 개입하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

아래의 `intervene_with_h` 함수를 완성해야 합니다. 여기에는 다음 과정이 포함됩니다:

* zero-shot 데이터셋에 대해 (동일한 context manager 내에서) 두 번의 forward pass를 실행합니다:
    * 개입이 없는 경우 (즉, residual stream이 변경되지 않음),
    * `h`을 사용하여 개입한 경우 (즉, `h`가 추출되었던 layer의 residual stream에 추가됨).
* 개입이 없는 경우와 개입한 경우의 completion을 각각 반환합니다 (docstring 참조).

아래 다이어그램은 `calculate_h` 함수와 결합되었을 때 이 모든 과정이 어떻게 작동하는지 보여줍니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/h-intervention-2.png" width="950">

힌트 - `tokenizer.batch_decode`을 사용하여 token 리스트를 문자열 리스트로 변환할 수 있습니다.

<details>
<summary>도움말 - 개입이 없는 경우와 개입한 경우의 completion을 모두 얻는 가장 좋은 방법을 모르겠습니다.</summary>

배치에 추가하기 위해 동일한 context manager 내에서 `with tracer.invoke...`을 여러 번 사용할 수 있습니다. 이렇게 하면 최종적으로 (2*N, seq_len) 형태의 출력을 얻게 되며, 이를 인덱싱하고 reshape 하여 개입이 없는 경우와 개입한 경우의 completion을 각각 얻을 수 있습니다.

</details>

<details>
<summary>도움말 - hidden state에 어떻게 개입해야 할지 모르겠습니다.</summary>

먼저, (이전에 했던 것처럼 `.output[0]`을 사용하여) hidden state 텐서를 정의할 수 있습니다.

그 다음, 이 텐서에 직접 더하거나 (또는 인덱싱된 버전에 더하거나) 할 수 있습니다. inplace 연산(즉, `tensor += h`)을 사용하거나 텐서를 재정의(즉, `tensor = tensor + h`)할 수 있으며, 두 방법 모두 가능합니다.

</details>

In [ ]:
def intervene_with_h(
    model: LanguageModel,
    zero_shot_dataset: ICLDataset,
    h: Tensor,
    layer: int,
    remote: bool = REMOTE,
) -> tuple[list[str], list[str]]:
    """
    Extracts the vector `h` using previously defined function, and intervenes by adding `h` to the
    residual stream of a set of generated zero-shot prompts.

    Inputs:
        model: the model we're using to generate completions
        zero_shot_dataset: the dataset of zero-shot prompts which we'll intervene on, using the
            `h`-vector
        h: the `h`-vector we'll be adding to the residual stream
        layer: the layer we'll be extracting the `h`-vector from
        remote: whether to run the forward pass on the remote server (used for running test code)

    Returns:
        completions_zero_shot: list of string completions for the zero-shot prompts, without
            intervention using the h-vector
        completions_intervention: list of string completions for the zero-shot prompts, with
            intervention using the h-vector
    """
    raise NotImplementedError()


tests.test_intervene_with_h(intervene_with_h, model, h, ANTONYM_PAIRS, REMOTE)

<details><summary>솔루션</summary>

```python
def intervene_with_h(
    model: LanguageModel,
    zero_shot_dataset: ICLDataset,
    h: Tensor,
    layer: int,
    remote: bool = REMOTE,
) -> tuple[list[str], list[str]]:
    """
    Extracts the vector `h` using previously defined function, and intervenes by adding `h` to the
    residual stream of a set of generated zero-shot prompts.

    Inputs:
        model: the model we're using to generate completions
        zero_shot_dataset: the dataset of zero-shot prompts which we'll intervene on, using the
            `h`-vector
        h: the `h`-vector we'll be adding to the residual stream
        layer: the layer we'll be extracting the `h`-vector from
        remote: whether to run the forward pass on the remote server (used for running test code)

    Returns:
        completions_zero_shot: list of string completions for the zero-shot prompts, without
            intervention using the h-vector
        completions_intervention: list of string completions for the zero-shot prompts, with
            intervention using the h-vector
    """
    with model.trace(remote=remote) as tracer:
        # First, run a forward pass where we don't intervene, just save token id completions
        with tracer.invoke(zero_shot_dataset.prompts):
            token_completions_zero_shot = model.lm_head.output[:, -1].argmax(dim=-1).save()

        # Next, run a forward pass on the zero-shot prompts where we do intervene
        with tracer.invoke(zero_shot_dataset.prompts):
            # Add the h-vector to the residual stream, at the last sequence position
            hidden_states = model.transformer.h[layer].output[0]
            hidden_states[:, -1] += h
            # Also save completions
            token_completions_intervention = model.lm_head.output[:, -1].argmax(dim=-1).save()

    # Decode to get the string tokens
    completions_zero_shot = model.tokenizer.batch_decode(token_completions_zero_shot)
    completions_intervention = model.tokenizer.batch_decode(token_completions_intervention)

    return completions_zero_shot, completions_intervention
```
</details>

함수의 completion을 계산하기 위해 아래 코드를 실행하십시오.

**주의: zero shot 데이터셋에 대해 서로 다른 random seed를 설정하는 것이 매우 중요합니다. 그렇지 않으면 $h$을 계산하는 데 사용한 데이터셋에 실제로 포함되어 있던 예시들에 대해 intervention을 수행하게 됩니다!**

In [ ]:
layer = 12
dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=3, seed=0)
zero_shot_dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=0, seed=1)

# Run previous function to get h-vector
h = calculate_h(model, dataset, layer=layer)[1]

# Run new function to intervene with h-vector
completions_zero_shot, completions_intervention = intervene_with_h(model, zero_shot_dataset, h, layer=layer)

print("Zero-shot completions: ", completions_zero_shot)
print("Completions with intervention: ", completions_intervention)

다음으로, 아래 코드를 실행하여 completion 결과들을 테이블로 시각화합니다. 다음과 같은 결과를 확인하실 수 있습니다:

* intervention이 없는 zero-shot prompt의 경우, 모델이 보통 prompt의 첫 번째이자 유일한 단어를 그대로 복사하기 때문에 정답률이 0%(또는 0에 가까움)로 나타납니다.
* intervention이 적용된 zero-shot prompt의 경우, 25-50%의 정답률을 보입니다.

In [ ]:
def display_model_completions_on_h_intervention(
    dataset: ICLDataset,
    completions: list[str],
    completions_intervention: list[str],
    num_to_display: int = 20,
) -> None:
    table = Table(
        "Prompt",
        "Model's completion\n(no intervention)",
        "Model's completion\n(intervention)",
        "Correct completion",
        title="Model's antonym completions",
    )

    for i in range(min(len(completions), num_to_display)):
        completion_ni = completions[i]
        completion_i = completions_intervention[i]
        correct_completion = dataset.completions[i]
        correct_completion_first_token = tokenizer.tokenize(correct_completion)[0].replace("Ġ", " ")
        seq = dataset.seqs[i]

        # Color code the completion based on whether it's correct
        is_correct = completion_i == correct_completion_first_token
        completion_i = f"[b green]{repr(completion_i)}[/]" if is_correct else repr(completion_i)

        table.add_row(str(seq), repr(completion_ni), completion_i, repr(correct_completion))

    rprint(table)


display_model_completions_on_h_intervention(zero_shot_dataset, completions_zero_shot, completions_intervention)

### 연습 문제 - 마지막 두 함수 결합하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

`nnsight` 라이브러리의 훌륭한 특징 중 하나는 forward pass를 병렬화하고 단일 context manager 내에서 복잡한 intervention을 수행할 수 있는 능력입니다.

위의 코드에서는 모델에서 hidden state를 추출하는 함수 하나와, 해당 hidden state에 intervention을 수행하는 또 다른 함수가 있었습니다. 하지만 실제로는 이 두 가지를 동시에 수행할 수 있습니다. 동일한 `model.trace` context manager 내에서, forward pass 중에 $h$를 계산하고, 이후 다른 forward pass(zero-shot prompt 사용)에서 이를 통해 intervention을 수행할 수 있습니다. 다시 말해, **이 context manager 내에서 `with tracer.invoke...`를 세 번 사용하게 됩니다.**

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/h-intervention-3.png" width="1000">

이를 위해 아래의 `calculate_h_and_intervene` 함수를 완성해야 합니다. 주로 `calculate_h` 함수와 `intervene_with_h` 함수를 결합하고, forward pass들을 동일한 context manager로 감싸는 작업(그리고 약간의 코드 수정)이 포함될 것입니다.

(`ICLDataset` 클래스는 deterministic하므로) 출력 결과는 이전과 정확히 동일해야 합니다. 따라서 이번에는 테스트 함수를 제공하지 않았으며, 얻은 테이블을 이전 테이블과 비교해 보시면 됩니다! 하지만 이번에는 "$h$ 계산"과 "$h$을 이용한 intervention" 작업을 단일 forward pass로 batching하여 처리하므로, 코드가 두 배 더 빠르게 실행될 것입니다.

<details>
<summary>도움말 - context manager 내부에서 <code>h</code> 벡터를 어떻게 사용하는지 잘 모르겠습니다.</summary>

`h`은 이전과 동일한 방식으로 추출하지만, 이를 저장할 필요는 없습니다. 이는 proxy로 유지됩니다. 실제 tensor인 것처럼 context manager 내에서 나중에 계속 사용할 수 있습니다.

token completion 외에는 context manager 내부에서 그 어떤 것도 `.save()` 해서는 안 됩니다.

</details>
<details>
<summary>도움말 - hidden state tensor <code>h</code>의 슬라이스에 <code>x</code> 벡터를 더하고 싶을 때, <code>h[slice]+=x</code>와 <code>h2 = h[slice], h2 += x</code>가 동일한가요?</summary>

아니요, `h[slice]+=x`만이 의도하신 대로 동작합니다. <code>h2 = h[slice], h2 += x</code>를 수행할 때, 수정 라인인 <code>h2 += x</code>는 더 이상 원본 tensor `h`를 수정하는 것이 아니라 다른 tensor`h2`를 수정하기 때문입니다. 반면, `h[slice]+=x`은 수정 라인에서 원본 tensor `h`를 그대로 유지합니다.

기억해 두면 좋은 규칙은 다음과 같습니다. in-place operation으로 tensor를 수정하려는 경우, 해당 tensor가 실제 수정 라인에 있는지 확인하십시오!

</details>

In [ ]:
def calculate_h_and_intervene(
    model: LanguageModel,
    dataset: ICLDataset,
    zero_shot_dataset: ICLDataset,
    layer: int,
) -> tuple[list[str], list[str]]:
    """
    Extracts the vector `h`, intervenes by adding `h` to the residual stream of a set of generated
    zero-shot prompts, all within the same forward pass. Returns the completions from this
    intervention.

    Inputs:
        model: LanguageModel
            the model we're using to generate completions
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the `h`-vector
        zero_shot_dataset: ICLDataset
            the dataset of zero-shot prompts which we'll intervene on, using the `h`-vector
        layer: int
            the layer we'll be extracting the `h`-vector from

    Returns:
        completions_zero_shot: list[str]
            list of string completions for the zero-shot prompts, without intervention
        completions_intervention: list[str]
            list of string completions for the zero-shot prompts, with h-intervention
    """
    raise NotImplementedError()


dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=3, seed=0)
zero_shot_dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=0, seed=1)

completions_zero_shot, completions_intervention = calculate_h_and_intervene(
    model, dataset, zero_shot_dataset, layer=layer
)

display_model_completions_on_h_intervention(zero_shot_dataset, completions_zero_shot, completions_intervention)

<details><summary>솔루션</summary>

```python
def calculate_h_and_intervene(
    model: LanguageModel,
    dataset: ICLDataset,
    zero_shot_dataset: ICLDataset,
    layer: int,
) -> tuple[list[str], list[str]]:
    """
    Extracts the vector `h`, intervenes by adding `h` to the residual stream of a set of generated
    zero-shot prompts, all within the same forward pass. Returns the completions from this
    intervention.

    Inputs:
        model: LanguageModel
            the model we're using to generate completions
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the `h`-vector
        zero_shot_dataset: ICLDataset
            the dataset of zero-shot prompts which we'll intervene on, using the `h`-vector
        layer: int
            the layer we'll be extracting the `h`-vector from

    Returns:
        completions_zero_shot: list[str]
            list of string completions for the zero-shot prompts, without intervention
        completions_intervention: list[str]
            list of string completions for the zero-shot prompts, with h-intervention
    """
    with model.trace(remote=REMOTE) as tracer:
        with tracer.invoke(dataset.prompts):
            h = model.transformer.h[layer].output[0][:, -1].mean(dim=0)

        with tracer.invoke(zero_shot_dataset.prompts):
            clean_tokens = model.lm_head.output[:, -1].argmax(dim=-1).save()

        with tracer.invoke(zero_shot_dataset.prompts):
            hidden = model.transformer.h[layer].output[0]
            hidden[:, -1] += h
            intervene_tokens = model.lm_head.output[:, -1].argmax(dim=-1).save()

    completions_zero_shot = tokenizer.batch_decode(clean_tokens)
    completions_intervention = tokenizer.batch_decode(intervene_tokens)
    return completions_zero_shot, completions_intervention
```
</details>

### 연습 문제 - 정확도 변화 계산하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

지금까지 우리는 가장 가능성이 높은 completion들을 살펴보고, 이것들이 얼마나 자주 정답이었는지 확인했습니다. 하지만 forward pass는 단순히 token completion만 제공하는 것이 아니라, logit 또한 제공합니다!

이제 `calculate_h_and_intervene` 함수를 수정하여, 두 개의 문자열 completion 리스트를 반환하는 대신, 각각 intervention이 없는 경우와 있는 경우에 대해 **모델이 정답 antonym에 할당한 logprob**를 포함하는 두 개의 float 리스트를 반환하도록 만들어야 합니다.

<details>
<summary>도움말 - logit에서 정답 logprob를 어떻게 가져오는지 모르겠습니다.</summary>

첫째, logit에 log softmax를 적용하여 logprob를 구합니다.

둘째, `tokenizer(dataset.completions)["input_ids"]`를 사용하여 정답 completion의 token ID를 가져올 수 있습니다. (주의 - 일부 단어는 여러 개의 token으로 tokenized될 수 있으므로, 각 completion에 대해 첫 번째 token ID만 선택하도록 하십시오.)

참고 - 이 모든 과정을 context manager 내부에서 수행한 다음, 모든 logit이 아닌 정답 logprob만 저장하고 반환하는 것을 권장합니다 (이렇게 하면 서버에서 다운로드할 양이 줄어듭니다!).

</details>

In [ ]:
def calculate_h_and_intervene_logprobs(
    model: LanguageModel,
    dataset: ICLDataset,
    zero_shot_dataset: ICLDataset,
    layer: int,
) -> tuple[list[float], list[float]]:
    """
    Extracts the vector `h`, intervenes by adding `h` to the residual stream of a set of generated
    zero-shot prompts, all within the same forward pass. Returns the logprobs on correct tokens from
    this intervention.

    Inputs:
        model: LanguageModel
            the model we're using to generate completions
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the `h`-vector
        zero_shot_dataset: ICLDataset
            the dataset of zero-shot prompts which we'll intervene on, using the `h`-vector
        layer: int
            the layer we'll be extracting the `h`-vector from

    Returns:
        correct_logprobs: list[float]
            list of correct-token logprobs for the zero-shot prompts, without intervention
        correct_logprobs_intervention: list[float]
            list of correct-token logprobs for the zero-shot prompts, with h-intervention
    """
    raise NotImplementedError()

<details><summary>솔루션</summary>

```python
def calculate_h_and_intervene_logprobs(
    model: LanguageModel,
    dataset: ICLDataset,
    zero_shot_dataset: ICLDataset,
    layer: int,
) -> tuple[list[float], list[float]]:
    """
    Extracts the vector `h`, intervenes by adding `h` to the residual stream of a set of generated
    zero-shot prompts, all within the same forward pass. Returns the logprobs on correct tokens from
    this intervention.

    Inputs:
        model: LanguageModel
            the model we're using to generate completions
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the `h`-vector
        zero_shot_dataset: ICLDataset
            the dataset of zero-shot prompts which we'll intervene on, using the `h`-vector
        layer: int
            the layer we'll be extracting the `h`-vector from

    Returns:
        correct_logprobs: list[float]
            list of correct-token logprobs for the zero-shot prompts, without intervention
        correct_logprobs_intervention: list[float]
            list of correct-token logprobs for the zero-shot prompts, with h-intervention
    """
    correct_completion_ids = [toks[0] for toks in tokenizer(zero_shot_dataset.completions)["input_ids"]]

    with model.trace(remote=REMOTE) as tracer:
        with tracer.invoke(dataset.prompts):
            h = model.transformer.h[layer].output[0][:, -1].mean(dim=0)

        with tracer.invoke(zero_shot_dataset.prompts):
            clean_logprobs = model.lm_head.output.log_softmax(dim=-1)[
                range(len(zero_shot_dataset)), -1, correct_completion_ids
            ].save()

        with tracer.invoke(zero_shot_dataset.prompts):
            hidden = model.transformer.h[layer].output[0]
            hidden[:, -1] += h
            intervene_logprobs = model.lm_head.output.log_softmax(dim=-1)[
                range(len(zero_shot_dataset)), -1, correct_completion_ids
            ].save()

    return clean_logprobs, intervene_logprobs
```
</details>

아래 코드를 실행하면 log-probabilities가 표시됩니다 (zero-shot 케이스보다 증가한 경우 초록색으로 강조됩니다). 모든 시퀀스에서 정답 token의 logprobs가 intervention 시에 증가하는 것을 확인할 수 있습니다. 이는 한 가지 중요한 점을 명확히 해줍니다. **최대 가능도(maximum-likelihood) token이 바뀌지 않더라도, 이것이 intervention이 유의미한 효과를 내지 못하고 있음을 의미하는 것은 아닙니다.**

In [ ]:
def display_model_logprobs_on_h_intervention(
    dataset: ICLDataset,
    correct_logprobs_zero_shot: list[float],
    correct_logprobs_intervention: list[float],
    num_to_display: int = 20,
) -> None:
    table = Table(
        "Zero-shot prompt",
        "Model's logprob\n(no intervention)",
        "Model's logprob\n(intervention)",
        "Change in logprob",
        title="Model's antonym logprobs, with zero-shot h-intervention\n(green = intervention improves accuracy)",
    )

    for i in range(min(len(correct_logprobs_zero_shot), num_to_display)):
        logprob_ni = correct_logprobs_zero_shot[i]
        logprob_i = correct_logprobs_intervention[i]
        delta_logprob = logprob_i - logprob_ni
        zero_shot_prompt = f"{dataset[i].x[0]:>8} -> {dataset[i].y[0]}"

        # Color code the logprob based on whether it's increased with this intervention
        is_improvement = delta_logprob >= 0
        delta_logprob = f"[b green]{delta_logprob:+.2f}[/]" if is_improvement else f"{delta_logprob:+.2f}"

        table.add_row(zero_shot_prompt, f"{logprob_ni:.2f}", f"{logprob_i:.2f}", delta_logprob)

    rprint(table)


dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=3, seed=0)
zero_shot_dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=0, seed=1)

correct_logprobs_zero_shot, correct_logprobs_intervention = calculate_h_and_intervene_logprobs(
    model, dataset, zero_shot_dataset, layer=layer
)

display_model_logprobs_on_h_intervention(
    zero_shot_dataset, correct_logprobs_zero_shot, correct_logprobs_intervention
)

# 3️⃣ Function Vectors

> ##### 학습 목표
>
> * in-context learning 태스크의 정확한 성능에 대해 각 attention head가 미치는 인과적 효과(causal effect)를 측정하는 지표를 정의합니다.
> * 특정 attention head에 해당하는 activation을 추출하기 위해, `nnsight` forward pass 동안 모델 내의 activation을 재배치하는 방법을 이해합니다.
> * multi-token 생성을 위해 `nnsight` 을 사용하는 방법을 배웁니다.

이 섹션에서는 모델의 ICL 성능에 큰 영향을 미치는 attention head 세트를 식별하고, 무작위로 섞인 prompt에서도 태스크 해결 동작을 유도하기 위해 이 벡터들로 patch할 수 있음을 보여줌으로써 논문 결과의 핵심 내용을 재현해 보겠습니다.

또한 multi-token generation을 위해 `nnsight`를 사용하는 방법과 모델의 동작을 steer하는 방법을 배울 것입니다. 예를 들어 Country-Capitals 태스크와 같이 다양한 태스크에 대해 이를 시도해 볼 수 있는 연습 문제들이 있으며, 여기서는 Amsterdam에 대해 이야기함으로써 `"When you think of Netherlands, you usually think of"`와 같은 prompt를 완성하도록 모델을 steer할 수 있습니다.

참고 - 이 섹션은 구조적으로 function vectors 논문의 섹션 2.2, 2.3 및 섹션 3의 일부 내용을 따릅니다.

여기서는 residual stream 상태에 대한 생각에서 **특정 attention head의 출력**에 대한 생각으로 넘어가겠습니다.

## FV 추출 및 사용하기

### `out_proj`에 관한 참고 사항

먼저, 약간의 기술적인 복잡함이 있습니다. 대부분의 HuggingFace 모델은 깔끔한 attention head 표현을 가지고 있지 않습니다. 우리가 가진 것은 "attention head별 projection"과 "attention head에 대한 합산" 연산을 암시적으로 결합한 linear layer `out_proj` 입니다 (이것이 어떻게 가능한지 이해가 가지 않는다면, Anthropic의 [Mathematical Framework](https://transformer-circuits.pub/2021/framework/index.html)에 있는 "Attention Heads are Independent and Additive" 섹션을 참조하십시오).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/rearrange-output-2.png" width="950">

이는 attention head에 대한 causal intervention을 수행할 때 몇 가지 의문을 제기합니다. 아래의 드롭다운을 사용하여 질문을 읽고 답해 보십시오 (이 내용은 이어지는 실습에서 중요하게 다뤄집니다).

<br>

<details>
<summary>특정 head에 대해 causal intervention을 수행하고 싶다면, <code>z</code> (<code>out_proj</code>의 입력)에 개입해야 할까요, 아니면 <code>attn_output</code> (<code>out_proj</code>의 출력)에 개입해야 할까요?</summary>

`z`에 개입해야 합니다. 왜냐하면 `(batch, seq, d_model)` 모양의 `z` tensor를 `(batch, seq, n_heads, d_head)`로 재배치하여, 다시 말해 모든 head를 분리해낼 수 있기 때문입니다. 반면에 `attn_output`의 경우에는 이미 head들에 대해 합산되었기 때문에 이를 다시 분리할 수 없습니다.

</details>

<br>

<details>
<summary>만약 context manager 내에서 모델 weight에 접근할 수 있는 능력이 있다면, 단일 head에 대한 <code>attn_output</code> 벡터를 어떻게 얻을 수 있을까요?</summary>

단일 attention head에 해당하는 `z` tensor의 슬라이스를 가져올 수 있습니다:

```python
z.reshape(batch, seq, n_heads, d_head)[:, :, head_idx]
```

그리고 단일 attention head에 해당하는 `out_proj` weight matrix의 슬라이스를 가져올 수 있습니다 (PyTorch는 linear layer를 `(out_feats, in_feats)` 모양으로 저장한다는 점을 기억하십시오):

```python
out_proj.weight.rearrange(d_model, n_heads, d_head)[:, head_idx]
```

마지막으로 이 둘을 곱하면 됩니다.

</details>

<br>

<details>
<summary>만약 context manager 내에서 모델 weight에 접근할 수 있는 능력이 </b>없다면</b>, 단일 head에 대한 <code>attn_output</code> 벡터를 어떻게 얻을 수 있을까요? (현재 <code>nnsight</code>가 이러한 경우인데, weight에 접근할 수 있게 되면 사용자가 이를 변경할 수 있기 때문입니다!).</summary>

약간의 기교를 부려, output projection을 통과시키기 전에 `z` 벡터에서 특정 head들을 ablate 할 수 있습니다:

```python
# ablate all heads except #2 (using a cloned activation)
heads_to_ablate = [0, 1, 3, 4, ...]
z_ablated = z.reshape(batch, seq, n_heads, d_head).clone()
z_ablated[:, :, heads_to_ablate] = 0

# save the output
attn_head_output = out_proj(z_ablated).save()
```

설명:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/rearrange-output-ablated-2.png" width="950">

참고 - 만약 `out_proj`에 bias가 있다면 이 방법은 실패할 것입니다. 왜냐하면 우리는 bias 항이 아니라 attention head의 출력만을 얻고 싶기 때문입니다. 하지만 [documentation page](https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html)을 확인해 보면 `out_proj`에는 bias 항이 없으므로 문제없습니다!

</details>

### 연습 문제 - `calculate_fn_vectors_and_intervene` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 30-60 minutes on this exercise.
> ```

이 함수는 아마도 오늘 연습 문제에서 가장 중요한 함수일 것입니다. 구현 방식은 이전 함수인 `calculate_h_and_intervene`와 매우 비슷하겠지만, 다음과 같은 차이점이 있습니다:

* 특정 layer의 residual stream `h` 값을 추출하는 대신, attention head의 출력을 추출하게 됩니다. 즉, 모델의 각 layer와 각 head를 반복하며 확인해야 합니다.
    * 모든 값을 계산하기 위해 clean forward pass는 한 번만 실행하면 되지만, 각 head에 대해서는 별도의 corrupted forward pass를 실행해야 합니다.
* 두 개의 서로 다른 데이터셋이 (dataset, zero-shot dataset)이었던 것과 달리, 여기서는 (dataset, 해당 데이터셋의 corrupted 버전)이 됩니다.
    * 이를 위해 `ICLDataset` 클래스의 `create_corrupted_dataset` 메서드를 사용할 수 있습니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/cie-intervention.png" width="1200">

실제로 코드를 작성하기 전에, 다음 질문에 답해 보는 것이 도움이 될 것입니다:

<details>
<summary>총 몇 번의 <code>invoke</code> 호출이 필요할까요?</summary>

`(N_LAYERS * N_HEADS) + 2`번이 필요합니다. 설명하자면 다음과 같습니다:

- 내부 activation을 추출하여 corrupted prompt에 패치하기 위한 clean prompt용 호출 한 번,
- 개입(intervene)하지 않는 corrupted prompt용 호출 한 번,
- **모든 attention head에 대해**, clean run activation을 사용하여 패치하기 위한 corrupted prompt용 호출 각각 한 번씩입니다.

</details>

<details>
<summary>이 함수에서 어떤 proxy output(있다면)에 <code>.save()</code>를 사용해야 할까요?</summary>

모델 내부에서 추출하는 function vector들은 context manager 내에서 causal intervention을 위해서만 사용되므로, `.save()` 할 필요가 없습니다.

저장해야 할 유일한 것은 (1) 개입하지 않는 corrupted forward pass와 (2) 하나의 head에 개입하는 각각의 corrupted forward pass에 대한 정답 token logprobs입니다. 다시 말해, 총 `(N_LAYERS * N_HEADS) + 1` 개의 tensor를 저장해야 합니다.

</details>

몇 가지 추가 참고 사항입니다:

* `layers` 인자를 추가하여 모델의 서로 다른 layer들을 반복해서 확인할 수 있도록 했습니다 (예를 들어, `layers = [3, 4, 5]`으로 모델을 실행하면 layer 3, 4, 5의 attention head에 대한 개입만 테스트합니다). 이는 모든 layer를 한 번에 실행할 때 메모리 오류가 발생하는 경우 유용합니다 (24개의 layer와 layer당 16개의 head가 있으므로, head당 prompt 수가 적더라도 금방 늘어납니다!).
    * 아래에 함수를 여러 번 호출하고, 각 실행 사이에 메모리를 비운 뒤 결과를 결합하는 방법을 보여주는 코드를 포함했습니다.
* 개입을 수행할 때, reshape된 tensor의 값을 설정할 수 있습니다. 즉, `tensor.reshape(*new_shape)[index] = new_value`는 `tensor`를 실제로 reshape하지 않고도 그 값을 변경합니다 (자세한 내용은 [`torch.Tensor.view`](https://pytorch.org/docs/stable/generated/torch.Tensor.view.html) 문서를 참조하십시오).
* shape이 예상과 일치하는지 확인하기 위해 코드에 많은 assert 문을 삽입하는 것이 좋은 습관입니다.
* 차원(dimension)이 헷갈린다면 `.reshape` 대신 `einops.rearrange`을 사용하십시오. 이는 실제 코드 내에서 코드 주석을 사용하는 것과 같은 매우 훌륭한 도구입니다!

마지막 참고 사항 - **계산 리소스 문제로 이 함수를 실행하는 것이 불가능하다면, 이 연습 문제를 건너뛰고 다음 문제로 넘어가셔도 됩니다. 다음 문제들은 이 함수의 작동 여부에 의존하지 않습니다.** 하지만 솔루션을 읽고 이해하는 과정은 반드시 거치시길 권장합니다.

In [ ]:
def calculate_fn_vectors_and_intervene(
    model: LanguageModel,
    dataset: ICLDataset,
    layers: list[int] | None = None,
) -> Float[Tensor, "layers heads"]:
    """
    Returns a tensor of shape (layers, heads), containing the CIE for each head.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the function vector (we'll also
            create a corrupted version of this dataset for interventions)
        layers: list[int] | None
            the layers which this function will calculate score for (if None, this means all layers)
    """
    raise NotImplementedError()

<details><summary>솔루션</summary>

```python
def calculate_fn_vectors_and_intervene(
    model: LanguageModel,
    dataset: ICLDataset,
    layers: list[int] | None = None,
) -> Float[Tensor, "layers heads"]:
    """
    Returns a tensor of shape (layers, heads), containing the CIE for each head.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the function vector (we'll also
            create a corrupted version of this dataset for interventions)
        layers: list[int] | None
            the layers which this function will calculate score for (if None, this means all layers)
    """
    layers = range(model.config.n_layer) if (layers is None) else layers
    heads = range(model.config.n_head)

    # Get corrupted dataset
    corrupted_dataset = dataset.create_corrupted_dataset()
    N = len(dataset)

    # Get correct token ids, so we can get correct token logprobs
    correct_completion_ids = [toks[0] for toks in tokenizer(dataset.completions)["input_ids"]]

    with model.trace(remote=REMOTE) as tracer:
        # Run a forward pass on clean prompts, where we store attention head outputs
        z_dict = {}
        with tracer.invoke(dataset.prompts):
            for layer in layers:
                # Get hidden states, reshape to get head dimension, store the mean tensor
                z = model.transformer.h[layer].attn.out_proj.input[:, -1]
                z_reshaped = z.reshape(N, N_HEADS, D_HEAD).mean(dim=0)
                for head in heads:
                    z_dict[(layer, head)] = z_reshaped[head]

        # Run a forward pass on corrupted prompts, where we don't intervene or store activations (just so we can get the
        # correct-token logprobs to compare with our intervention)
        with tracer.invoke(corrupted_dataset.prompts):
            logits = model.lm_head.output[:, -1]
            correct_logprobs_corrupted = logits.log_softmax(dim=-1)[t.arange(N), correct_completion_ids].save()

        # For each head, run a forward pass on corrupted prompts (here we need multiple different forward passes, since
        # we're doing different interventions each time)
        correct_logprobs_dict = {}
        for layer in layers:
            for head in heads:
                with tracer.invoke(corrupted_dataset.prompts):
                    # Get hidden states, reshape to get head dimension, then set it to the a-vector
                    z = model.transformer.h[layer].attn.out_proj.input[:, -1]
                    z.reshape(N, N_HEADS, D_HEAD)[:, head] = z_dict[(layer, head)]
                    # Get logprobs at the end, which we'll compare with our corrupted logprobs
                    logits = model.lm_head.output[:, -1]
                    correct_logprobs_dict[(layer, head)] = logits.log_softmax(dim=-1)[
                        t.arange(N), correct_completion_ids
                    ].save()

    # Get difference between intervention logprobs and corrupted logprobs, and take mean over batch dim
    all_correct_logprobs_intervention = einops.rearrange(
        t.stack([v for v in correct_logprobs_dict.values()]),
        "(layers heads) batch -> layers heads batch",
        layers=len(layers),
    )
    logprobs_diff = all_correct_logprobs_intervention - correct_logprobs_corrupted  # shape [layers heads batch]

    # Return mean effect of intervention, over the batch dimension
    return logprobs_diff.mean(dim=-1)
```
</details>

앞서 언급했듯이, 아래 코드는 함수를 여러 번 개별적으로 호출하고 그 결과들을 결합합니다.

이 코드를 실행하고 결과를 시각화하면, Function Vectors 논문의 Figure 3(a)와 거의 유사한 결과를 얻을 수 있습니다. 만약 코드 실행 시간이 너무 오래 걸린다면, 논문의 그림과 비교했을 때 뚜렷한 패턴을 보이는 단일 layer만 선택하여 실행하는 것을 권장합니다 (예를 들어, L8H1 head가 해당 layer의 다른 모든 head보다 훨씬 높은 점수를 가지는 layer 8이 적절합니다).

In [ ]:
dataset = ICLDataset(ANTONYM_PAIRS, size=8, n_prepended=2)


def batch_process_layers(n_layers, batch_size):
    for i in range(0, n_layers, batch_size):
        yield range(n_layers)[i : i + batch_size]


results = t.empty((0, N_HEADS), device=device)

# If this fails to run, you should reduce the batch size so the forward passes are split up more, or
# reduce dataset size
for layers in batch_process_layers(N_LAYERS, batch_size=4):
    print(f"Computing layers in {layers} ...")
    t0 = time.time()
    results = t.concat([results, calculate_fn_vectors_and_intervene(model, dataset, layers).to(device)])
    print(f"... finished in {time.time() - t0:.2f} seconds.\n")

In [ ]:
imshow(
    results.T,
    title="Average indirect effect of function-vector intervention on antonym task",
    width=1000,
    height=600,
    labels={"x": "Layer", "y": "Head"},
    aspect="equal",
)

### 연습 문제 - function vector 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 25-50 minutes on this exercise.
> ```

다음 과제는 실제로 function vector를 계산하여 반환하는 것입니다. 이를 통해 몇 가지 실험을 진행할 수 있습니다. function vector는 이전 함수를 통해 찾은 모든 attention head 출력값의 합(즉, 이 head들이 residual stream에 쓰는 모든 벡터의 합)을 데이터셋의 prompt들에 대해 평균 낸 값입니다.

여기에는 한 가지 어려움이 있습니다. 단순히 `z` 벡터를 얻는 것이 아니라, head들에 대해 합산되기 *전*의 `attn_out` 벡터를 얻으려고 하기 때문입니다. 이전에 논의했듯이, 우리가 사용하는 모델의 경우 `out_proj` linear map이 "project up"과 "sum over heads" 연산을 동시에 수행하므로 이를 구현하는 것이 다소 까다롭습니다. `out_proj` 행렬의 일부 슬라이스와 `z` 벡터의 일부 슬라이스를 곱하면 좋겠지만, `nnsight` 라이브러리는 (보안상의 이유로) 사용자가 weight에 직접 접근하는 것을 아직 허용하지 않습니다. 기본 weight에 접근하지 않고 개별 head에 대한 `attn_out` 벡터를 어떻게 추출할 수 있는지 이해하려면, 이 섹션 시작 부분의 **`out_proj`에 관한 참고 사항** 소섹션을 다시 읽어보시기 바랍니다.

In [ ]:
def calculate_fn_vector(
    model: LanguageModel,
    dataset: ICLDataset,
    head_list: list[tuple[int, int]],
) -> Float[Tensor, " d_model"]:
    """
    Returns a vector of length `d_model`, containing the sum of vectors written to the residual
    stream by the attention heads in `head_list`, averaged over all inputs in `dataset`.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the function vector (we'll also
            create a corrupted version of this dataset for interventions)
        head_list: list[tuple[int, int]]
            list of attention heads we're calculating the function vector from
    """
    raise NotImplementedError()


tests.test_calculate_fn_vector(calculate_fn_vector, model)

<details><summary>솔루션</summary>

```python
def calculate_fn_vector(
    model: LanguageModel,
    dataset: ICLDataset,
    head_list: list[tuple[int, int]],
) -> Float[Tensor, " d_model"]:
    """
    Returns a vector of length `d_model`, containing the sum of vectors written to the residual
    stream by the attention heads in `head_list`, averaged over all inputs in `dataset`.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the function vector (we'll also
            create a corrupted version of this dataset for interventions)
        head_list: list[tuple[int, int]]
            list of attention heads we're calculating the function vector from
    """
    # Turn head_list into a dict of {layer: heads we need in this layer}
    head_dict = defaultdict(set)
    for layer, head in head_list:
        head_dict[layer].add(head)

    fn_vector_list = []

    with model.trace(dataset.prompts, remote=REMOTE):
        for layer, head_list in head_dict.items():
            # Get the output projection layer
            out_proj = model.transformer.h[layer].attn.out_proj

            # Get the mean output projection input (note, setting values of this tensor will not
            # have downstream effects on other tensors)
            hidden_states = out_proj.input[:, -1].mean(dim=0)

            # Zero-ablate all heads which aren't in our list, then get the output (which
            # will be the sum over the heads we actually do want!)
            heads_to_ablate = set(range(N_HEADS)) - head_dict[layer]
            for head in heads_to_ablate:
                hidden_states.reshape(N_HEADS, D_HEAD)[head] = 0.0

            # Now that we've zeroed all unimportant heads, get the output & add it to the list
            # (we need a single batch dimension so we can use `out_proj`)
            out_proj_output = out_proj(hidden_states.unsqueeze(0)).squeeze()
            fn_vector_list.append(out_proj_output.save())

    # We sum all attention head outputs to get our function vector
    fn_vector = sum([v for v in fn_vector_list])

    assert fn_vector.shape == (D_MODEL,)
    return fn_vector
```
</details>

## Multi-token generation

이제 논문의 Table 3에 있는 일부 결과들을 재현해 보겠습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/tab3.png" width="700">

여기서는 이전에 하지 않았던 작업인 **multi-token prompt generation에 대한 개입(intervening)**을 수행하게 됩니다.

이 장의 대부분의 interpretability 연습 문제는 autoregressive text generation보다는 단일 forward pass를 실행하는 것으로 구성되었습니다. 하지만 여기서는 다른 시도를 해보겠습니다. 텍스트 생성 중 각 forward pass에서 최종 sequence position에 function vector를 추가하여, 모델이 다른 의미의 문장을 출력하게 만들 수 있는지 확인하는 것입니다.

Table 3의 결과는 원래 prompt의 최종 sequence position과 **이후 생성되는 각 토큰의 최종 sequence position**의 residual stream에 function vector를 추가하여 얻은 것입니다. 이렇게 하는 이유는 시간이 지남에 따라 모델의 행동을 유도하기 위해서입니다. 우리의 가설은 function vector가 "다음 토큰 반의어 행동(next-token antonym behaviour)"을 유도한다는 것입니다 (왜냐하면 이 벡터는 ICL prompt에서 모델이 반의어 예측을 하기 직전의 sequence position에서 attention head 출력값들을 평균 내어 계산되었기 때문입니다).

### Multi-token generation을 위한 `nnsight` 사용하기

이전까지 우리의 context manager는 다음과 같은 형태였습니다:

```python
# Single invoke
with model.trace(prompt, remote=REMOTE):
    ... # Intervene on fwd pass

# Multiple invokes
with model.trace(remote=REMOTE) as tracer:
    with tracer.invoke(prompt):
        ... # Intervene on fwd pass
```

하지만 multi-token generation의 경우, `trace` 대신 `generate` 메서드를 사용할 것입니다. 우리의 context manager는 다음과 같은 형태가 됩니다:

```python
# Single invoke
with model.generate(prompt, remote=REMOTE, max_new_tokens=max_new_tokens):
    for n in range(max_new_tokens):
        ... # Intervene on fwd pass for n-th token to be generated
        model.next()

# Multiple invokes
with model.generate(max_new_tokens=max_new_tokens, remote=REMOTE) as generator:
    with generator.invoke(prompt):

        for n in range(max_new_tokens):
            ... # Intervene on fwd pass for n-th token to be generated
            model.next()
```

`model.next()` 라인은 이후의 개입들이 *다음* 토큰을 생성하는 forward pass에 적용되어야 함을 나타냅니다.

대부분의 경우, single-token generation 동안 배운 모든 내용은 multi-token 케이스로 일반화됩니다. 예를 들어, `.save()`을 사용하는 것은 여전히 context manager 외부로 proxy를 저장합니다 (다만, 서로 다른 생성 과정에서 동일한 변수 이름을 사용하지 않도록 주의하십시오. 그렇지 않으면 덮어쓰게 됩니다. 저장된 proxy들을 리스트나 dict 등에 저장하는 것이 더 쉽습니다).

`model.generate`은 일반적인 [HuggingFace generate method](https://huggingface.co/docs/transformers/en/main_classes/text_generation)와 동일한 인자를 받는다는 점에 유의하십시오. 이는 `top_k`, `top_p`, 또는 `repetition_penalty`와 같은 인자를 사용하여 생성 행동을 제어할 수 있음을 의미합니다. 아래 연습 문제에서는 repetition penalty를 사용합니다 ([paper](https://arxiv.org/pdf/1909.05858)의 제안에 따라 1.2 값을 선택했습니다). 이는 모델이 동일한 시퀀스를 반복하는 루프에 빠지는 것을 방지할 수 있으며, 특히 모델을 OOD로 밀어내는 steering 과정에서 흔히 발생합니다.

<!-- #### Optional questions - multi-token generation with NNsight

Here are a few quick optional questions to test your understanding of how multi-generation works with NNsight. These are non-essential, and only mentioned here as potentially helpful pointers.  


<details>
<summary>원래 prompt의 모든 토큰에는 벡터 <code>h</code>를 추가하고, 생성된 토큰에는 추가하지 않으려면 어떻게 해야 합니까? </summary>

```python
with model.generate(max_new_tokens=max_new_tokens, remote=REMOTE) as generator:
    with generator.invoke(prompt):
        # Add vectors to the model's internals on the first forward pass
        model.transformer.h[layer].output[0][:, :seq_len] += h

```
원래 prompt의 토큰들에 벡터를 한 번만 추가하는 것이므로 `model.next()`을 호출할 필요가 없습니다. 이는 모델이 이후에 토큰을 생성할 때 캐싱됩니다.

</details>

<details>
<summary>처음 k개의 생성된 토큰을 생성하는 동안 벡터 <code>h</code>로 개입하려면 어떻게 해야 합니까? </summary>

처음 `k`개의 생성된 토큰 생성 중에 개입하려면 다음과 같이 합니다:
```python
with model.generate(max_new_tokens=max_new_tokens, remote=REMOTE) as generator:
    with generator.invoke(prompt):

        for n in range(k+1):
            # Add vector to the model's internals, on the k-th forward pass
            model.transformer.h[layer].output[0] += h
            model.next()
```
`n=0`일 때, 새로운 토큰이 생성되기 전 원래 prompt의 토큰들에 추가하는 것입니다. `model.next()`을 호출한 후에는, 방금 생성된 마지막 토큰의 hidden state에 접근하게 됩니다 (seq_len=1).

</details>

</details>

<details>
<summary>원래 prompt의 토큰들에는 추가하지 않고, 오직 처음 k개의 토큰을 생성하는 동안에만 벡터 <code>h</code>로 개입하려면 어떻게 해야 합니까? </summary>

```python
with model.generate(max_new_tokens=max_new_tokens, remote=REMOTE) as generator:
    with generator.invoke(prompt):

        for n in range(k+1):
            model.next()
            # Add vector AFTER calling model.next() to add to the token that just got generated
            model.transformer.h[layer].output[0] += h

```
`model.next()` 이전에 아무것도 추가하지 않음으로써, 원래 prompt에는 절대 추가하지 않고 항상 새로운 토큰이 생성된 후에만 추가하게 됩니다.

</details>

</details>

<details>
<summary>벡터 <code>h</code>를 <code>model.next()</code> 전과 후에 추가하는 것의 차이점은 무엇입니까? </summary>

Q3에서 설명했듯이, `model.next()` 전에 벡터를 추가하는 것은 새로운 생성 토큰이 추가되기 **전**에 항상 현재 시퀀스에 대해 연산이 수행됨을 의미합니다. `model.next()` 후에 벡터를 추가하는 것은 항상 새로 생성된 토큰에 대해 연산이 수행됨을 의미합니다.

</details> -->

### Key-Value Caching

요약 - 캐싱은 `model.generate` 내부의 causal intervention을 더 복잡하게 만들 수 있지만, 이는 아주 마지막 sequence position이 아닌 다른 위치에 개입할 때만 해당됩니다. 우리의 연습 문제에서는 마지막 seqpos에만 개입할 것이므로 걱정하실 필요 없으나, 여전히 이해해두면 유용한 주제입니다.

<details>
<summary>더 자세한 내용이 궁금하시면 이 드롭다운을 확인하십시오.</summary>

추론 속도를 높이기 위해, transformer 모델은 텍스트 생성을 가속화하는 **key-value caching**을 수행합니다. 이는 $n$개의 토큰을 생성하는 데 걸리는 시간이 단일 토큰을 생성하는 시간의 $n$배보다 ***훨씬*** 적게 걸림을 의미합니다. transformer 추론 연산에 대한 자세한 내용은 [this blog post](https://kipp.ly/transformer-inference-arithmetic/)을 참조하십시오.

캐싱이 발생하고 causal intervention을 수행할 때, 캐싱이 우리의 causal intervention을 덮어쓰지 않도록 주의해야 합니다. 때로는 causal intervention이 올바르게 작동하도록 하기 위해 캐싱을 비활성화해야 할 때가 있습니다. 예를 들어, "생성하는 각 토큰에 대해 prompt의 *오직* 최종 sequence position에만 function vector를 추가"하는 개입을 수행하고 싶다면, 캐싱을 비활성화해야 합니다 (이전 forward pass들이 더 이상 최종 sequence position이 아닌 위치에 개입했던 캐싱된 값들을 포함하고 있기 때문입니다). 하지만 여기서는 "원래 prompt의 최종 토큰과 *이후의 모든 sequence position*에 function vector를 추가"하는 개입을 수행하므로, 캐싱을 활성화하는 것(기본 동작)이 올바른 causal intervention 결과를 줍니다.

</details>

### Generator Output

`generator.output` 객체는 기본적으로 모델의 token ID completion을 포함하는 tensor입니다 (logit이 아닙니다).

기본적으로 `generate` 메서드는 greedily하게 토큰을 생성합니다. 즉, 매 단계에서 항상 최대 확률을 가진 토큰을 선택합니다. 지금은 이 동작을 변경하는 것에 대해 걱정할 필요가 없습니다. 하지만 향후 연습 문제에서는 greedy sampling(generate가 기본적으로 사용하는 방식) 외에 다른 sampling 메서드들을 실험할 것이므로, `generator.output`와 logit에 대해 argmax를 취하는 것이 동일하지 않을 것입니다!

### 연습 문제 - 멀티 토큰 생성에서 function vector로 개입하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> ```

이제 아래의 function `intervene_with_fn_vector` 을 완성해야 합니다. 이 함수는 (위에서 작성한 함수로부터 계산된) function vector와 몇 가지 다른 인자들(docstring 참조)을 입력으로 받아, 주어진 prompt template에 대한 모델의 문자열 completion을 반환합니다.

우리는 표 3에서와 같이, 모델이 특정 단어를 그 반의어로 정의하는 것과 같은 정성적인 결과를 관찰하기를 기대합니다.

In [ ]:
def intervene_with_fn_vector(
    model: LanguageModel,
    word: str,
    layer: int,
    fn_vector: Float[Tensor, " d_model"],
    prompt_template='The word "{x}" means',
    n_tokens: int = 5,
) -> tuple[str, str]:
    """
    Intervenes with a function vector, by adding it at the last sequence position of a generated
    prompt.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        word: str
            The word substituted into the prompt template, via prompt_template.format(x=word)
        layer: int
            The layer we'll make the intervention (by adding the function vector)
        fn_vector: Float[Tensor, "d_model"]
            The vector we'll add to the final sequence position for each new token to be generated
        prompt_template:
            The template of the prompt we'll use to produce completions
        n_tokens: int
            The number of additional tokens we'll generate for our unsteered / steered completions

    Returns:
        completion: str
            The full completion (including original prompt) for the no-intervention case
        completion_intervention: str
            The full completion (including original prompt) for the intervention case
    """
    raise NotImplementedError()

<details><summary>솔루션</summary>

```python
def intervene_with_fn_vector(
    model: LanguageModel,
    word: str,
    layer: int,
    fn_vector: Float[Tensor, " d_model"],
    prompt_template='The word "{x}" means',
    n_tokens: int = 5,
) -> tuple[str, str]:
    """
    Intervenes with a function vector, by adding it at the last sequence position of a generated
    prompt.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        word: str
            The word substituted into the prompt template, via prompt_template.format(x=word)
        layer: int
            The layer we'll make the intervention (by adding the function vector)
        fn_vector: Float[Tensor, "d_model"]
            The vector we'll add to the final sequence position for each new token to be generated
        prompt_template:
            The template of the prompt we'll use to produce completions
        n_tokens: int
            The number of additional tokens we'll generate for our unsteered / steered completions

    Returns:
        completion: str
            The full completion (including original prompt) for the no-intervention case
        completion_intervention: str
            The full completion (including original prompt) for the intervention case
    """
    prompt = prompt_template.format(x=word)

    with model.generate(remote=REMOTE, max_new_tokens=n_tokens, repetition_penalty=1.2) as generator:
        with model.all():
            with generator.invoke(prompt):
                tokens = model.generator.output.save()

            with generator.invoke(prompt):
                model.transformer.h[layer].output[0][:, -1] += fn_vector
                tokens_intervention = model.generator.output.save()

    completion, completion_intervention = tokenizer.batch_decode(
        [tokens.squeeze().tolist(), tokens_intervention.squeeze().tolist()]
    )
    return completion, completion_intervention
```
</details>

작성하신 함수를 테스트하려면 아래 코드를 실행하십시오. 첫 번째 completion은 정상적으로 보이지만, 두 번째 completion에서는 단어를 그 반의어로 정의하는 것을 확인할 수 있습니다 (출력의 효과와 일관성 사이의 균형을 맞추기 위해 `fn_vector`의 scale factor를 조금 조정해야 할 수도 있습니다). 이것이 제대로 작동한다면 축하드립니다 - **방금 6b-parameter 모델에서 OOD 행동 변화를 성공적으로 유도하신 것입니다!**

In [ ]:
# Remove word from our pairs, so it can be a holdout
word = "light"
_ANTONYM_PAIRS = [pair for pair in ANTONYM_PAIRS if word not in pair]

# Define our dataset, and the attention heads we'll use
dataset = ICLDataset(_ANTONYM_PAIRS, size=20, n_prepended=5)
head_list = [
    (8, 0),
    (8, 1),
    (9, 14),
    (11, 0),
    (12, 10),
    (13, 12),
    (13, 13),
    (14, 9),
    (15, 5),
    (16, 14),
]

# Extract the function vector
fn_vector = calculate_fn_vector(model, dataset, head_list)

# Intervene with the function vector
completion, completion_intervention = intervene_with_fn_vector(
    model,
    word=word,
    layer=9,
    fn_vector=1.5 * fn_vector,
    prompt_template='The word "{x}" means',
    n_tokens=40,
)

table = Table("No intervention", "intervention")
table.add_row(repr(completion), repr(completion_intervention))
rprint(table)

### 연습 문제 - 다른 태스크로 결과 일반화하기 (선택 사항)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> ```

이 연습 문제에서는 반의어(antonyms) 태스크와 다른 태스크를 직접 선택하여, (동일한 attention head 세트에 대해) 결과가 여전히 유효한지 확인합니다.

이 연습 문제는 여러분이 채워 넣어야 할 코드 템플릿 없이 상당히 개방적인 형태로 제공됩니다. 하지만 가이드가 필요하시다면 아래의 드롭다운 메뉴를 사용하실 수 있습니다.

<details>
<summary>연습 문제 가이드</summary>

어떤 태스크를 선택하시든, 새로운 단어 세트를 생성해야 합니다. 반의어 태스크에서 사용한 `generate_dataset` 함수를 활용하여 다른 prompt와 초기 예시 세트를 제공하거나(아직 생성하지 않으셨다면 OpenAI api key 생성 및 사용이 필요합니다), 온라인에서 적절한 데이터셋을 찾으실 수 있습니다.

`ICLDataset`를 정의할 때, 만약 선택한 태스크가 대칭적이지 않다면 `bidirectional=False`를 사용하고 싶으실 것입니다. 반의어 태스크는 대칭적이지만, 다른 태스크(예: 국가-수도 태스크)는 그렇지 않습니다.

`intervene_with_fn_vector` 함수를 위해 새로운 prompt 템플릿을 제공해야 하지만, 그 외의 대부분의 코드는 동일하게 유지될 것입니다.

</details>

In [ ]:
with open(section_dir / "data/country_capital_pairs.txt", "r", encoding="utf-8") as f:
    COUNTRY_CAPITAL_PAIRS = [line.split() for line in f.readlines()]

country = "Netherlands"
_COUNTRY_CAPITAL_PAIRS = [pair for pair in COUNTRY_CAPITAL_PAIRS if pair[0] != country]

dataset = ICLDataset(_COUNTRY_CAPITAL_PAIRS, size=20, n_prepended=5, bidirectional=False)
head_list = [
    (8, 0),
    (8, 1),
    (9, 14),
    (11, 0),
    (12, 10),
    (13, 12),
    (13, 13),
    (14, 9),
    (15, 5),
    (16, 14),
]

fn_vector = calculate_fn_vector(model, dataset, head_list)

# Intervene with the function vector
completion, completion_intervention = intervene_with_fn_vector(
    model=model,
    word=country,
    layer=9,
    fn_vector=fn_vector,
    prompt_template="When you think of {x},",
    n_tokens=40,
)

table = Table("No intervention", "intervention")
table.add_row(repr(completion), repr(completion_intervention))
rprint(table)

# 4️⃣ GPT2-XL의 Steering Vectors

> ##### 학습 목표
>
> * Alex Turner 등의 steering vectors 연구의 목표와 주요 결과에 대해 이해합니다.
> * 그들의 초기 포스트에서 설명된 행동 변화를 재현합니다.

**참고**: 현재 GPT2-XL은 NNsight에 의해 원격으로 호스팅되지 않습니다. GPT2-XL을 사용하시는 경우, `REMOTE = False` 설정을 권장합니다. 그렇지 않으면 원격으로 호스팅되는 모델 중 하나를 사용하고([here](https://nnsight.net/status/) 참조) `REMOTE = True`를 설정할 수 있습니다. 새로운 모델을 로드하기 전에 메모리를 확보하기 위해 `del model` 및 `gc.collect()`를 실행하는 것이 좋습니다.

## 모델 행동 제어하기

이전 섹션의 마지막 비보너스 연습 문제에서, 우리는 모델이 zero-shot 또는 손상된 prompt에 대해 올바른 completion을 생성하도록 만드는 것뿐만 아니라, function vector를 사용하여 모델의 completion에 행동 변화를 유도하는 아이디어를 다루었습니다. 다음 연습 문제들에서는 이러한 종류의 연구를 더 자세히 살펴보겠습니다. 주로 [Steering GPT-2-XL by adding an activation vector](https://www.lesswrong.com/posts/5spBue2z2tw4JuDCx/steering-gpt-2-xl-by-adding-an-activation-vector)에 관한 Turner et al의 연구를 사용할 것입니다.

이 연구가 지금까지 우리가 진행한 function vector 연구와 어떻게 다른지에 대한 요약입니다:

* Function vector는 모델이 특정 기능(예: 단어를 반대말로 매핑)을 수행하는 것에 집중한 반면, 이 연구는 행동 변화(예: 부정적인 톤의 prompt를 긍정적인 방식으로 completion 하는 것)에 집중합니다.
* Function vector 연구는 매우 큰 모델을 다루었습니다(우리의 연습 문제에서는 function vector 논문에서 조사된 가장 작은 모델인 Pythia-7B를 사용했습니다). 이 steering vector 포스트는 더 작은 모델인 GPT2-Small (85m)과 GPT2-XL (1.5B)에 집중합니다. 우리는 GPT2-XL에 집중할 것입니다.
* function vector 연구의 후반부에서는 중요한 attention head를 식별하고 residual stream에 직접 더하는 대신 해당 head의 output에 집중했습니다. 이 steering vector 설정에서는 residual stream에 직접 더하는 더 단순한 방법으로 돌아가겠습니다.

이러한 차이점에도 불구하고, 여기서 수행된 많은 작업은 function vector 연구와 겹칩니다. 두 가지 모두 *"forward-pass 기반 방법(즉, SGD가 아닌 방법)을 사용하여 벡터를 찾고, 이를 forward pass 중에 모델에 개입시켜 모델의 output을 변경하는 것"*이라는 더 넓은 범주에 속하기 때문입니다. 이 설명에는 다음 내용도 포함됩니다:

* [Inference-time intervention](https://www.lesswrong.com/posts/kuQfnotjkQA4Kkfou/inference-time-intervention-eliciting-truthful-answers-from): "모델이 진실을 말하게 만드는" 행동 변화를 유도하는 데 집중합니다. 또한 CCS 및 linear probing와 같이 intervention vector를 찾기 위한 다른 비-forward-pass 기반 기술들도 살펴봅니다. 하지만 지금까지 우리가 사용해 온 것과 유사한 forward-pass 기반 방법이 가장 효과적이라는 결론을 내립니다.
* [Steering Llama 2 via Contrastive Activation Addition](https://arxiv.org/abs/2312.06681): GPT2-XL steering vector 연구를 더 큰 모델, 특히 Llama 2 13B로 확장한 것으로 생각할 수 있습니다. 또한 sycophancy, myopia, power-seeking와 같은 속성의 변화를 측정하는 더 고수준의 evals 프레임워크를 취합니다(적절한 벡터를 추가함으로써 이러한 속성들을 증가시키거나 감소시킬 수 있음을 발견했습니다).

이 연구들 중 일부는 보너스 섹션에서 더 자세히 논의하겠지만, 지금은 연습 문제를 시작해 보겠습니다!

먼저 GPT2-XL을 로드한 다음, 메인 포스트에 있는 몇 가지 예제들을 재현해 보겠습니다.

In [ ]:
gpt2_xl = LanguageModel("gpt2-xl", device_map="auto", torch_dtype=t.bfloat16)
tokenizer = gpt2_xl.tokenizer

REMOTE = False
# If you are using gpt2_xl, set REMOTE = False as gpt2_xl is not hosted remotely by nnsight. You can
# set REMOTE = True for a remotely hosted model here (https://nnsight.net/status/)

### 연습 문제 - steering vector 결과 재현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 30-50 minutes on this exercise.
> ```

LessWrong 포스트 [Steering GPT-2-XL by adding an activation vector](https://www.lesswrong.com/posts/5spBue2z2tw4JuDCx/steering-gpt-2-xl-by-adding-an-activation-vector#fnrefcvnfx3e6sfu)의 결과, 특히 "demonstrations of additions that work well" 섹션을 재현합니다.

벡터가 어떻게 추출되고 추가되는지 이해하기 위해 [Steering GPT-2-XL by adding an activation vector](https://www.lesswrong.com/posts/5spBue2z2tw4JuDCx/steering-gpt-2-xl-by-adding-an-activation-vector#How_activation_additions_work)의 "How activation additions work" 섹션을 읽어보시기 바랍니다. 함수 템플릿과 실행을 위한 예제 코드를 제공해 드렸으며, 여러분의 주된 작업은 함수 내부를 채우는 것입니다. 이는 이전의 여러 연습 문제들의 하이브리드 형태가 될 것이며 (함수 `calculate_and_intervene_with_h`과 가장 유사합니다), 몇 가지 방법론적인 차이가 있을 것입니다.

이것이 이번 세트의 마지막 연습 문제이며, 지금까지 배운 모든 내용을 하나로 엮을 수 있는 기회가 되기를 바랍니다!

### Caching

이것은 이전 섹션에서 수행했던 것과는 다른 종류의 causal intervention입니다. 매 token 생성 시 마지막 sequence 위치에 단일 벡터를 추가하는 대신, 원래 prompt의 첫 sequence 위치들에 벡터 슬라이스를 추가합니다 (설명을 위해 [this section](https://www.lesswrong.com/posts/5spBue2z2tw4JuDCx/steering-gpt-2-xl-by-adding-an-activation-vector#1__Love___Hate)와 같은 표를 참조하십시오). 이것이 우리의 함수에 어떤 영향을 줄 것이라고 생각하십니까? 여전히 cache를 사용해야 할까요? `.generate()`를 사용해야 할까요, 아니면 `.trace()`을 사용해야 할까요? `.generate()`을 사용하는 경우, `model.next()`을 호출해야 할까요?

<details>
<summary>위 질문들에 대한 답변을 보려면 이 드롭다운을 클릭하십시오.</summary>

생성되는 모든 token에 대해 매번 마지막 sequence 위치에 추가하는 것이 아니라, prompt의 끝에 벡터를 한 번만 추가합니다. 이는 다음과 같음을 의미합니다:

- 여전히 caching을 사용할 수 있습니다 (cache하는 값들이 이후의 token 생성 과정에서 달라지지 않기 때문입니다).
- `.generate()`를 사용해야 합니다 (multi-token generation을 수행하기 때문입니다).
- `model.next()`을 호출할 필요가 없습니다 (단 한 번만 개입하며, 이 개입 내용이 cache되어 이후 생성되는 모든 token에 적용되기 때문입니다).

다시 한번 말씀드리지만, 이 내용 중 혼란스러운 부분이 있다면 TA에게 질문하거나 Slack 채널에 메시지를 남겨주시기 바랍니다.

</details>

### Padding

[tables](https://www.lesswrong.com/posts/5spBue2z2tw4JuDCx/steering-gpt-2-xl-by-adding-an-activation-vector#1__Love___Hate)에서는 activation이 왼쪽에서 추가되는 것으로 나타나 있습니다 (즉, sequence가 오른쪽에 padding됨). 하지만 기본적으로 padding은 왼쪽에 적용됩니다. 이를 해결할 수 있는 두 가지 방법이 있습니다:

1. 입력 sequence를 수동으로 right-pad 합니다. 즉, `len(tokenizer.tokenize(prompt))`과 같은 방법을 사용하여 각 prompt의 길이를 확인하고, 각 sequence의 끝에 `tokenizer.pad_token`의 복사본을 추가합니다.
2. 입력 sequence를 수동으로 pad 하지 않는 대신, 원래 prompt에 추가할 sequence를 activation addition sequence의 왼쪽이 아닌 오른쪽에서 슬라이싱합니다.

솔루션에서는 (2)번 방법을 사용하지만, 두 방법 중 어느 것을 사용해도 무방합니다.

### Sampling

포스트를 따라, sequence 생성을 위해 확률 0.3의 top-p sampling을 사용하겠습니다. 또한 반복을 방지하여 모델이 루프에 빠지는 일을 줄이기 위해 작은 frequency penalty를 사용하겠습니다. 이 섹션의 이전 연습 문제들을 수행했다면 sampling 중에 `freq_penalty`을 구현했을 수도 있습니다. 이는 TransformerLens 모델에서 지원되지만, HuggingFace는 이와 다소 유사한 `repetition_penalty`를 사용합니다 (기본값은 1.0으로 penalty가 없음을 의미하며, 1.0보다 큰 값은 반복되는 token에 penalty를 적용합니다).

이러한 sampling 방법들은 `generate` 메서드에 keyword argument를 전달하여 적용합니다:

```python
{
    "do_sample": True, # necessary whenever we're sampling rather than doing greedy decoding
    "top_p": 0.3,
    "repetition_penalty": 1.1,
}
```

sequence가 greedily가 아니라 stochastically 생성된다는 점에 유의하십시오. 이는 동일한 sequence를 여러 번 입력하더라도 서로 다른 결과가 나올 수 있음을 의미합니다. 아래 함수에 `n_comparisons` 인자를 제공해 드렸으므로, steered completion과 unsteered completion을 각각 이 횟수만큼 생성해야 합니다.

### 기타 팁 / 참고 사항

예제 #9 ("talking about weddings" 예제)부터 시작하는 것을 권장합니다. `Love - Hate` 예제와 달리 forward pass의 정확한 조건에 상당히 강건한 것으로 보입니다. 아래에 제공된 템플릿 셀 중 어느 것이든 사용하실 수 있습니다.

`use_bos` 인자를 제공해 드렸습니다. 이 값이 True인 경우, 모든 prompt의 시작 부분에 `tokenizer.bos_token`를 추가해야 합니다. 이는 LessWrong 포스트의 구현 방식을 그대로 따르기 위함입니다. 동작에 큰 변화를 주지는 않으므로, 이를 무시하더라도 좋은 결과를 얻을 수 있을 것입니다.

In [ ]:
SAMPLING_KWARGS = {
    "do_sample": True,
    "top_p": 0.3,
    "repetition_penalty": 1.2,
}


def calculate_and_apply_steering_vector(
    model: LanguageModel,
    prompt: str,
    activation_additions: list[tuple[int, float, str]],
    n_tokens: int,
    n_comparisons: int = 1,
    use_bos: bool = True,
) -> tuple[list[str], list[str]]:
    """
    Performs the steering vector experiments described in the LessWrong post.

    Args:
        model: LanguageModel
            the transformer you're doing this computation with
        prompt: str
            The original prompt, which we'll be doing activation steering on.

        activation_additions: list[tuple[int, float, str]], each tuple contains:
            layer - the layer we're applying these steering vectors to
            coefficient - the value we're multiplying it by
            prompt - the prompt we're inputting
            e.g. activation_additions[0] = [6, 5.0, " Love"] means we add the " Love" vector at
            layer 6, scaled by 5x

        n_tokens: int
            Number of tokens which will be generated for each completion

        n_comparisons: int
            Number of sequences generated in this function (i.e. we generate `n_comparisons` which
            are unsteered, and the same number which are steered).

    Returns:
        unsteered_completions: list[str]
            List of length `n_comparisons`, containing all the unsteered completions.

        steered_completions: list[str]
            List of length `n_comparisons`, containing all the steered completions.
    """
    # Add the BOS token manually, if we're including it
    if use_bos:
        bos = model.tokenizer.bos_token
        prompt = bos + prompt
        activation_additions = [[layer, coeff, bos + p] for layer, coeff, p in activation_additions]

    raise NotImplementedError()

<details><summary>솔루션</summary>

```python
SAMPLING_KWARGS = {
    "do_sample": True,
    "top_p": 0.3,
    "repetition_penalty": 1.2,
}


def calculate_and_apply_steering_vector(
    model: LanguageModel,
    prompt: str,
    activation_additions: list[tuple[int, float, str]],
    n_tokens: int,
    n_comparisons: int = 1,
    use_bos: bool = True,
) -> tuple[list[str], list[str]]:
    """
    Performs the steering vector experiments described in the LessWrong post.

    Args:
        model: LanguageModel
            the transformer you're doing this computation with
        prompt: str
            The original prompt, which we'll be doing activation steering on.

        activation_additions: list[tuple[int, float, str]], each tuple contains:
            layer - the layer we're applying these steering vectors to
            coefficient - the value we're multiplying it by
            prompt - the prompt we're inputting
            e.g. activation_additions[0] = [6, 5.0, " Love"] means we add the " Love" vector at
            layer 6, scaled by 5x

        n_tokens: int
            Number of tokens which will be generated for each completion

        n_comparisons: int
            Number of sequences generated in this function (i.e. we generate `n_comparisons` which
            are unsteered, and the same number which are steered).

    Returns:
        unsteered_completions: list[str]
            List of length `n_comparisons`, containing all the unsteered completions.

        steered_completions: list[str]
            List of length `n_comparisons`, containing all the steered completions.
    """
    # Add the BOS token manually, if we're including it
    if use_bos:
        bos = model.tokenizer.bos_token
        prompt = bos + prompt
        activation_additions = [[layer, coeff, bos + p] for layer, coeff, p in activation_additions]

    # Get the (layers, coeffs, prompts) in an easier form to use, also calculate the prompt lengths
    # and check they're all the same
    act_add_layers, act_add_coeffs, act_add_prompts = zip(*activation_additions)
    act_add_seq_lens = [len(tokenizer.tokenize(p)) for p in act_add_prompts]
    assert len(set(act_add_seq_lens)) == 1, "All activation addition prompts must be the same length."
    assert act_add_seq_lens[0] <= len(tokenizer.tokenize(prompt)), (
        "All act_add prompts should be shorter than original prompt."
    )

    # Get the prompts we'll intervene on (unsteered and steered)
    steered_prompts = [prompt for _ in range(n_comparisons)]
    unsteered_prompts = [prompt for _ in range(n_comparisons)]

    with model.generate(max_new_tokens=n_tokens, remote=REMOTE, **SAMPLING_KWARGS) as generator:
        # Run the act_add prompts (i.e. the contrast pairs), and extract their activations
        with generator.invoke(act_add_prompts):
            # Get all the prompts from the activation additions, and put them in a list
            # (note, we slice from the end of the sequence because of left-padding)
            act_add_vectors = [
                model.transformer.h[layer].output[0][i, -seq_len:]
                for i, (layer, seq_len) in enumerate(zip(act_add_layers, act_add_seq_lens))
            ]

        # Forward pass on unsteered prompts (no intervention, no activations saved - we only need
        # the completions)
        with generator.invoke(unsteered_prompts):
            unsteered_out = model.generator.output.save()

        # Forward pass on steered prompts (we add in the results from the act_add prompts)
        with generator.invoke(steered_prompts):
            # For each act_add prompt, add the vector to residual stream, at the start of the seq
            for i, (layer, coeff, seq_len) in enumerate(zip(act_add_layers, act_add_coeffs, act_add_seq_lens)):
                model.transformer.h[layer].output[0][:, :seq_len] += coeff * act_add_vectors[i]
            steered_out = model.generator.output.save()

    # Decode steered & unsteered completions (discarding the sequences we only used for extracting
    # activations) & return results
    unsteered_completions = tokenizer.batch_decode(unsteered_out[-n_comparisons:])
    steered_completions = tokenizer.batch_decode(steered_out[-n_comparisons:])

    return unsteered_completions, steered_completions
```
</details>

작성하신 함수를 테스트하려면 다음 코드 스니펫 중 하나를 사용하십시오 (앞서 언급했듯이, 결과가 상당히 견고하게 나오는 경향이 있는 weddings 예제로 시작하시는 것을 권장합니다).

In [ ]:
unsteered_completions, steered_completions = calculate_and_apply_steering_vector(
    gpt2_xl,
    prompt="I hate you because",
    activation_additions=[(6, +8.0, "Love "), (6, -8.0, "Hate")],
    n_tokens=50,
    n_comparisons=3,
    use_bos=True,
)

table = Table("Unsteered", "Steered", title="Completions", show_lines=True)
for usc, sc in zip(unsteered_completions, steered_completions):
    table.add_row(usc, sc)
rprint(table)

In [ ]:
unsteered_completions, steered_completions = calculate_and_apply_steering_vector(
    gpt2_xl,
    prompt="I went up to my friend and said",
    activation_additions=[
        (20, +4.0, "I talk about weddings constantly  "),
        (20, -4.0, "I do not talk about weddings constantly"),
    ],
    n_tokens=50,
    n_comparisons=3,
    use_bos=False,
)

table = Table("Unsteered", "Steered", title="Completions", show_lines=True)
for usc, sc in zip(unsteered_completions, steered_completions):
    table.add_row(usc, sc)
rprint(table)

In [ ]:
unsteered_completions, steered_completions = calculate_and_apply_steering_vector(
    gpt2_xl,
    prompt="To see the eiffel tower, people flock to",
    activation_additions=[
        (24, +10.0, "The Eiffel Tower is in Rome"),
        (24, -10.0, "The Eiffel Tower is in France"),
    ],
    n_tokens=50,
    n_comparisons=3,
    use_bos=False,
)

table = Table("Unsteered", "Steered", title="Completions", show_lines=True)
for usc, sc in zip(unsteered_completions, steered_completions):
    table.add_row(usc, sc)
rprint(table)

# ☆ 보너스

## Function Vectors 논문의 확장 내용

해당 논문에는 두 가지 다른 흥미로운 결과가 더 있지만, 지금까지 다룬 내용만큼 중요하지는 않습니다. 시간이 되신다면, 이 결과들을 직접 재현해 보시기 바랍니다.

### Function Vector의 디코딩된 어휘 (3.2)

이 섹션에서 저자들은 function vector의 디코딩된 어휘에서 상위 단어들(즉, unembedding vector와 function vector의 내적이 가장 높은 단어들)을 찾아내며, 이 단어들이 해당 태스크와 개념적으로 관련되어 있음을 보여줍니다. 예를 들어:

* antonyms 태스크의 경우, 상위 단어들은 `" negate"`, `" counterpart"`, `" lesser"`와 같이 반의어라는 개념을 떠올리게 합니다.
* country-capitals 태스크의 경우, 상위 단어들은 실제로 `" Moscow"`, `" Paris"`, `" Madrid"`와 같은 수도의 이름들입니다.

antonyms 태스크와 이전 섹션에서 선택한 태스크 모두에 대해 이러한 결과들을 재현해 보시겠습니까?

흥미로운 확장 질문입니다 - 만약 Country-Capitals 태스크(본질적으로 비대칭적임)와 같은 태스크를 수행하면서, function vector는 해당 태스크의 대칭 버전(즉, 각 질문-답변 쌍이 서로 뒤바뀔 수 있는 버전)에서 가져온다면 어떤 일이 발생할까요? 여전히 동일한 행동 결과가 나타나는지, 그리고 디코딩된 어휘 결과는 어떻게(혹은 어떻게 다르게) 변하는지 확인해 보십시오.

In [ ]:
# YOUR CODE HERE - find the decoded vocabulary

<details>
<summary>My results for this (spoiler!)</summary>

In the Country-Capitals task, I found:

* The bidirectional task does still work to induce behavioural changes, although slightly less effectively than for the original task.
* The top decoded vocabulary items are a mix of country names and capital names, but mostly capitals.

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">Top logits:
' London'
' Moscow'
' Madrid'
' Budapest'
' Athens'
' Paris'
' Berlin'
' Bangkok'
' Istanbul'
' Montreal'
' Barcelona'
' Jerusalem'
' Seoul'
' Miami'
' Dublin'
' Atlanta'
' Copenhagen'
' Mumbai'
' Minneapolis'
' Beijing'</pre>

</details>


<details><summary>Solution</summary>

```python
# Code to calculate decoded vocabulary:
logits = model._model.lm_head(fn_vector)
max_logits = logits.topk(20).indices.tolist()
tokens = model.tokenizer.batch_decode(max_logits)
print("Top logits:\n" + "\n".join(map(repr, tokens)))
```
</details>

### 함수 벡터의 벡터 대수 (3.3)

이 섹션에서 저자들은 함수 벡터가 합성될 수 있는지 조사합니다. 예를 들어, 어떤 의미에서 세 개의 개별적인 ICL 태스크가 합성되어 네 번째 태스크를 만든다고 할 때, 처음 세 태스크의 함수 벡터를 모두 더해 이를 네 번째 태스크의 함수 벡터로 사용할 수 있을까요?

저자들은 이를 다양한 태스크에서 테스트합니다. 그 결과 일부 태스크(예: 국가-수도 태스크, 여기서는 함수 벡터보다 성능이 더 좋습니다)에서는 효과적이지만, 일반적으로는 함수 벡터만큼 효과적이지 않다는 것을 발견합니다. 여러분도 동일한 결과를 얻으셨나요?

## Steering Vectors 포스트의 확장 과제

우리는 steering vectors 포스트의 결과 중 아주 작은 일부만 구현했습니다 (그마저도 상당히 급하게 처리했습니다). 하지만 여러분이 직접 시도해 볼 수 있는 다른 내용들이 많이 있습니다. 예를 들어:

* 저자들은 "프랑스어로 말하기" 벡터를 찾는 데 실패했다고 언급했습니다. LessWrong 포스트의 상위 댓글 중 하나에는 실제로 작동했던 프랑스어 벡터 생성 과정이 설명되어 있습니다 (댓글 링크 [here](https://www.lesswrong.com/posts/5spBue2z2tw4JuDCx/steering-gpt-2-xl-by-adding-an-activation-vector?commentId=sqsS9QaDy2bG83XKP)). 이 결과를 재현할 수 있습니까? (해당 댓글에 Colab 링크도 포함되어 있어, 진행 중 막힐 때 도움이 될 것입니다.)
* 논문의 [later section](https://www.lesswrong.com/posts/5spBue2z2tw4JuDCx/steering-gpt-2-xl-by-adding-an-activation-vector#Perplexity_on_lots_of_sentences_about_weddings_or_about_shipping)에서 저자들은 perplexity(entropy와 관련된 측정 지표)에 대해 광범위하게 논의합니다. 그들은 "weddings" 벡터가 결혼 관련 문장의 perplexity를 낮추고, 관련 없는 문장의 perplexity는 유지한다는 것을 발견했습니다. 이 결과, 특히 결혼 관련 문장과 비관련 문장에 대해 injection layer에 따른 perplexity 비율 그래프를 재현할 수 있습니까?
* 저자들은 이 포스트를 정식 논문으로 작성했으며, [here](https://arxiv.org/abs/2308.10248)에서 확인하실 수 있습니다. 이 논문에 담긴 추가 결과 중 일부를 재현할 수 있습니까?

## 추천 논문 재현 과제

### [Inference-Time Intervention: Eliciting Truthful Answers from a Language Model](https://arxiv.org/abs/2306.03341)

이 논문에서 저자들은 "모델이 진실을 말하게 만드는" 행동 변화를 유도하는 데 집중합니다. 또한 CCS나 linear probing와 같이 forward-pass 기반이 아닌 intervention vector를 찾는 다른 기술들도 살펴봅니다. 하지만 결과적으로는 우리가 지금까지 사용해 온 방식과 유사한 forward-pass 기반 방법들이 가장 효과적이라는 결론을 내립니다.

다음과 같은 경우에 이 재현 과제가 적합할 수 있습니다:

* 이 섹션의 연습 문제들이 즐거웠으며, 이 섹션에서 다루지 않은 기술(예: linear probing)을 실험하는 데 관심이 있는 경우,
* `nnsight` 라이브러리 등을 통해 매우 큰 모델을 다루는 것이 익숙한 경우,
* [model truthfulness](https://arxiv.org/abs/2109.07958) 연구에 관심이 있는 경우.

### [Steering Llama 2 via Contrastive Activation Addition](https://arxiv.org/abs/2312.06681)

이 논문은 GPT2-XL steering vector 연구를 더 큰 모델, 특히 Llama 2 13B로 확장한 것으로 볼 수 있습니다. 또한 더 고차원적인 evals 프레임워크를 채택하여 sycophancy, myopia, power-seeking와 같은 속성의 변화를 측정합니다 (적절한 vector를 추가함으로써 이러한 속성들을 증가시키거나 감소시킬 수 있음을 발견했습니다).

다음과 같은 경우에 이 재현 과제가 적합할 수 있습니다:

* 이 섹션의 연습 문제들이 즐거웠으며, 이러한 아이디어들을 태스크 기반의 맥락보다는 행동적 맥락에서 적용해보고 싶은 경우,
* `nnsight` 라이브러리 등을 통해 매우 큰 모델을 다루는 것이 익숙한 경우,
* myopia, power seeking 등과 같은 특성에 대한 [evaluating models](https://www.alignmentforum.org/posts/yRAo2KEGWenKYZG9K/discovering-language-model-behaviors-with-model-written)에 관심이 있는 경우,
* prompt-engineering과 대규모 데이터셋(위에 링크된 데이터셋 등)을 다루는 것이 익숙한 경우.

*업데이트* - 이 논문과 관련된 [LessWrong post](https://www.lesswrong.com/posts/v7f8ayBxLhmMFRzpa/steering-llama-2-with-contrastive-activation-additions)이 새로 게시되었으며, 여기에는 관련 분야에 대한 짧은 논의도 포함되어 있습니다. 이 재현 과제나 이 섹션의 다른 추천 재현 과제에 관심이 있다면 이 포스트를 읽어보실 것을 강력히 권장합니다.

### [Red-teaming language models via activation engineering](https://www.alignmentforum.org/posts/iHmsJdxgMEWmAfNne/red-teaming-language-models-via-activation-engineering)

Nina Rimsky가 수행한 이 연구는 우리가 이전에 보았던 많은 연구 결과들을 확장하여 **refusal** 도메인에 적용했습니다. 즉, LLM이 사용자의 요청에 응답하기를 거부할지 여부를 결정하는 요소는 무엇이며, 이 행동에 어떻게 영향을 줄 수 있는지를 다룹니다. 그녀의 포스트에서 발췌한 내용은 다음과 같습니다:

> *finetuning과 RLHF가 의도한 결과를 견고하게 달성했는지 검증하는 것은 어렵습니다... 우리는 수많은 입력을 검색하는 대신, inference 중에 내부 상태를 조작함으로써 모델에서 원치 않는 행동을 더 효율적으로 유발할 수 있습니다. activation engineering과 같은 기술을 통해 행동이 쉽게 유발될 수 있다면, 실제 배포 환경에서도 발생할 수 있다는 아이디어입니다. 작은 내부 섭동(perturbations)을 통해 행동을 이끌어낼 수 없다는 점은 더 강력한 안전 보장이 될 수 있습니다.*

다음과 같은 경우에 이 재현 과제가 적합할 수 있습니다:

* 이 섹션의 연습 문제들이 즐거웠으며, 이러한 아이디어들을 태스크 기반의 맥락보다는 행동적 맥락에서 적용해보고 싶은 경우,
* `nnsight` 라이브러리 등을 통해 매우 큰 모델을 다루는 것이 익숙한 경우,
* RLHF, adversarial attacks 및 jailbreaking에 관심이 있는 경우,
* prompt-engineering이 익숙한 경우 (이 재현 과제에 필요한 일부 데이터는 Nina의 [GitHub repo](https://github.com/nrimsky/LM-exp/tree/main)에서 확인할 수 있습니다).


<br>

---

<br>

참고 - 일주일 정도의 작업 기간으로는, 이러한 논문 재현 과제들을 시도하지 않는 것을 약하게 권장합니다. 왜냐하면 (참가자들이 `nnsight` 라이브러리를 사용할 수 있다는 점을 고려하더라도) 연산 비용이 상당히 많이 들기 때문입니다. function vector나 GPT2-XL 연구를 바탕으로 수행할 수 있는 다양한 재현 및 확장 가능성이 있으며, 이 섹션의 연습 문제들이 즐거웠고 그와 유사한 작업을 더 해보고 싶다면 이쪽이 더 나은 선택지가 될 수 있습니다.

하지만, 큰 모델을 다루는 것이 익숙하시거나(예: 관련 경험이 있는 경우) 이 연구에 관심이 있으시다면, 당연히 이러한 재현 과제들에 도전하시는 것을 환영합니다!